# Pixels to Pairs — Authoritative Master Benchmark (Corrected FUNSD Spatial Gold)

**Master revision:** corrected FUNSD spatial Gold + audited few-shot demos + 5-model final quantitative set + disk-safe resumable execution.

# From Pixels to Pairs — Final Readable Batch Benchmark

This notebook is the clean final implementation for the new benchmark reruns.

FUNSD, SROIE, and CORD each live inside a normal Python factory function. This gives every dataset its own lexical scope, preventing `TEXT_DIR`, `RESULTS_BASE`, prompts, parsers, and other variables from overwriting another dataset's configuration.

Final settings:
- corrected Value F1 implementation;
- `MAX_NEW_TOKENS = 1024`;
- `BATCH_SIZE = 1`;
- `RESUME = True`;
- five zero-shot benchmark models;
- Qwen2.5-7B and LLaMA-3-8B for the few-shot ablation cells;
- separate final-result directories;
- secure Hugging Face authentication;
- high-precision per-document metric CSVs.

Run the notebook top to bottom. Do not launch the full queue until the pre-flight cell passes.

**Disk-safe revision:** local Hugging Face caches are now managed model-by-model to prevent Colab VM disk exhaustion.


## 1. Mount Google Drive

### Updated final generation/parsing protocol

- `MAX_NEW_TOKENS = 1024` for every dataset/model condition.
- Full valid JSON is parsed normally.
- If a response is malformed or truncated, only complete, independently valid JSON `{"key": ..., "value": ...}` objects literally present in the output are retained.
- No regex KVP recovery, malformed-JSON repair, guessing, fuzzy parsing, GT-aware recovery, or document-text fallback is used.
- Per-document logs include `generated_tokens` and `output_hit_max_new_tokens`.

**Do not resume a prior 512-token result folder into a 1024-token run.** Delete/rename the old condition folder first, or temporarily set `RESUME=False`.


### FUNSD zero-target exclusion

For the KVP benchmark, FUNSD documents with **zero annotated ground-truth KVP pairs** are excluded from metric computation because Key Recall, Exact Match, and Value F1 are undefined when there are no targets. The exclusion is based on `len(gt_kvp) == 0`, not on document IDs or model performance, and is applied identically to Gold, Tesseract, EasyOCR, and PaddleOCR conditions.

In the inspected FUNSD test split, this rule excludes the three blank/no-answer forms `85201976`, `85540866`, and `85629964`, leaving **47 evaluable documents**. Original dataset files are left unchanged.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 2. Hugging Face authentication

In [ ]:
import os
from getpass import getpass

hf_token = os.environ.get("HF_TOKEN")

if not hf_token:
    try:
        from google.colab import userdata
        hf_token = userdata.get("HF_TOKEN")
    except Exception:
        hf_token = None

if not hf_token:
    hf_token = getpass("Enter your Hugging Face token: ").strip()

if hf_token:
    os.environ["HF_TOKEN"] = hf_token
    print("HF authentication configured.")
else:
    print("No HF token provided; only public checkpoints can be loaded.")

## 3. Record environment

In [ ]:
import platform
import torch
import transformers

print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA runtime:", torch.version.cuda)
    print("BF16 supported:", torch.cuda.is_bf16_supported())

## 4. Metric sanity checks

In [ ]:
from collections import Counter
import numpy as np

def _test_normalize_str(s: str) -> str:
    return " ".join(str(s).strip().lower().split())

def _test_text_f1(gt: str, pred: str) -> float:
    gt_tokens = _test_normalize_str(gt).split()
    pred_tokens = _test_normalize_str(pred).split()

    if not gt_tokens and not pred_tokens:
        return 1.0
    if not gt_tokens or not pred_tokens:
        return 0.0

    gt_c = Counter(gt_tokens)
    pred_c = Counter(pred_tokens)
    overlap = sum((gt_c & pred_c).values())

    if overlap == 0:
        return 0.0

    precision = overlap / len(pred_tokens)
    recall = overlap / len(gt_tokens)
    return 2 * precision * recall / (precision + recall)

def _test_compute_doc_metrics(gt_kvp, pred_kvp):
    num_gt = len(gt_kvp)
    if num_gt == 0:
        return {
            "key_recall": 0.0,
            "exact_match_rate": 0.0,
            "value_f1": 0.0,
        }

    gt_keys = [_test_normalize_str(x.get("key", "")) for x in gt_kvp]
    gt_vals = [_test_normalize_str(x.get("value", "")) for x in gt_kvp]
    pred_keys = [_test_normalize_str(x.get("key", "")) for x in pred_kvp]
    pred_vals = [_test_normalize_str(x.get("value", "")) for x in pred_kvp]

    gt_counts = Counter(k for k in gt_keys if k)
    pred_counts = Counter(k for k in pred_keys if k)
    matched_keys = sum(min(gt_counts[k], pred_counts.get(k, 0)) for k in gt_counts)
    key_recall = matched_keys / num_gt

    used_exact = set()
    exact_matches = 0
    for gk, gv in zip(gt_keys, gt_vals):
        if not gk or not gv:
            continue
        for j, (pk, pv) in enumerate(zip(pred_keys, pred_vals)):
            if j in used_exact:
                continue
            if gk == pk and gv == pv:
                exact_matches += 1
                used_exact.add(j)
                break
    exact_match_rate = exact_matches / num_gt

    used_f1 = set()
    value_scores = []
    for gt_pair in gt_kvp:
        gk = _test_normalize_str(gt_pair.get("key", ""))
        gv = str(gt_pair.get("value", ""))

        if not gk or not _test_normalize_str(gv):
            value_scores.append(0.0)
            continue

        best_j = None
        best_f1 = 0.0
        for j, pred_pair in enumerate(pred_kvp):
            if j in used_f1:
                continue
            pk = _test_normalize_str(pred_pair.get("key", ""))
            if not pk or pk != gk:
                continue
            f1 = _test_text_f1(gv, str(pred_pair.get("value", "")))
            # Assign the first same-key prediction even when F1 == 0.
            if best_j is None or f1 > best_f1:
                best_j = j
                best_f1 = f1

        if best_j is None:
            value_scores.append(0.0)
        else:
            used_f1.add(best_j)
            value_scores.append(best_f1)

    return {
        "key_recall": key_recall,
        "exact_match_rate": exact_match_rate,
        "value_f1": float(np.mean(value_scores)),
    }

tests = [
    (
        "perfect",
        [{"key": "Total", "value": "$100.00"}],
        [{"key": "Total", "value": "$100.00"}],
        (1.0, 1.0, 1.0),
    ),
    (
        "same key, zero value overlap",
        [{"key": "Total", "value": "$100.00"}],
        [{"key": "Total", "value": "ABC"}],
        (1.0, 0.0, 0.0),
    ),
    (
        "missing key",
        [{"key": "Total", "value": "$100.00"}],
        [],
        (0.0, 0.0, 0.0),
    ),
    (
        "one exact + one missing",
        [
            {"key": "Total", "value": "$100.00"},
            {"key": "Date", "value": "01/01/2026"},
        ],
        [{"key": "Total", "value": "$100.00"}],
        (0.5, 0.5, 0.5),
    ),
]

for name, gt, pred, expected in tests:
    result = _test_compute_doc_metrics(gt, pred)
    actual = (
        result["key_recall"],
        result["exact_match_rate"],
        result["value_f1"],
    )
    assert np.allclose(actual, expected), f"{name}: expected {expected}, got {actual}"
    print(f"PASS: {name} -> {actual}")

print("\\nAll metric sanity checks passed.")


## 5. Dataset runners

### FUNSD runner definition

### Key normalization used for evaluation

Field labels are compared with a formatting-tolerant normalizer: Unicode/case differences, surrounding decoration, common punctuation, and separator spacing are ignored. The normalizer does **not** use fuzzy matching, synonym expansion, stemming, or semantic repair. The model output itself is preserved unchanged, and the JSON parser remains strict.


In [ ]:
from types import SimpleNamespace

def build_funsd_runner():
    # =====================================================
    # Pixels to Pairs benchmark — FUNSD
    # Final benchmark runner
    # =====================================================
    # This cell preserves the recovered experimental prompt/parsing/generation
    # logic for FUNSD, while cleaning secrets/paths and using the corrected,
    # manuscript-aligned metric implementation.
    #
    # Before running:
    #   1) Mount Google Drive using the setup cell above.
    #   2) Set HF_TOKEN in the Colab environment if a gated model requires it.
    #   3) Check ENGINE, SHOT_VARIANT, MODEL_SPECS, and dataset paths.
    #   4) Use the experiment cells later in this notebook; they run configurations sequentially.
    #
    import os
    import json
    import gc
    from typing import Dict, List, Any, Optional, Tuple
    from collections import Counter

    import numpy as np
    import torch
    from transformers import (
        AutoTokenizer,
        AutoModelForSeq2SeqLM,
        AutoModelForCausalLM,
        AutoConfig,
    )
    from tqdm.auto import tqdm

    # =====================================================
    # 0. ENV (CUDA ALLOC)
    # =====================================================
    # Helps reduce fragmentation-related OOMs
    os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

    # =====================================================
    # 1. REPRODUCIBILITY (LIGHT)
    # =====================================================

    SEED = 0
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)

    # =====================================================
    # 2. HF TOKEN & PERSISTENT CACHE
    # =====================================================
    import os
    HF_TOKEN = os.environ.get("HF_TOKEN")
    HF_CACHE_DIR = "/root/.cache/huggingface"
    os.makedirs(HF_CACHE_DIR, exist_ok=True)

    os.environ["HF_HOME"] = HF_CACHE_DIR
    os.environ["TRANSFORMERS_CACHE"] = HF_CACHE_DIR
    print(f"Using HF cache dir: {HF_CACHE_DIR}")

    # =====================================================
    # 3. CONFIG – PATHS & GLOBALS
    # =====================================================

    BASE_DIR       = "/content/drive/MyDrive/data/funsd"
    ANNOTATION_DIR = os.path.join(BASE_DIR, "annotations")

    ENGINE = "gold_text_spatial"
    if ENGINE == "gold_text_spatial":
        TEXT_DIR = os.path.join(BASE_DIR, "funsd_gold_text_spatial")
    elif ENGINE == "gold_text":
        TEXT_DIR = os.path.join(BASE_DIR, "funsd_gold_text")
    elif ENGINE == "tesseract":
        TEXT_DIR = os.path.join(BASE_DIR, "funsd_ocr/tesseract")
    elif ENGINE == "paddleocr":
        TEXT_DIR = os.path.join(BASE_DIR, "funsd_ocr/paddleocr")
    elif ENGINE == "easyocr":
        TEXT_DIR = os.path.join(BASE_DIR, "funsd_ocr/easyocr")
    else:
        raise ValueError(f"Unknown ENGINE={ENGINE}")

    SHOT_VARIANT = "0shot"  # 0shot/1shot/2shot/3shot
    if SHOT_VARIANT not in {"0shot", "1shot", "2shot", "3shot"}:
        raise ValueError(f"Unsupported SHOT_VARIANT={SHOT_VARIANT}, use one of 0/1/2/3.")

    RESUME = True

    RESULTS_BASE = "/content/drive/MyDrive/data/pixels_to_pairs_final_results/funsd"
    os.makedirs(RESULTS_BASE, exist_ok=True)

    MAX_DOCS         = None
    MAX_NEW_TOKENS   = 1024  # fixed across all final benchmark runs
    MAX_INPUT_TOKENS = 8192

    # Keep BATCH_SIZE=1 for stability and easier debugging
    BATCH_SIZE       = 1

    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    DEBUG_FIRST_DOC = False

    # Stopping criteria disabled (paper-defensible default)
    ENABLE_STOPPING = False  # kept for compatibility; not used in this no-bnb version


    # =====================================================
    # 4. MODEL LIST – EDIT THIS TO CHOOSE MODELS
    # =====================================================

    MODEL_SPECS = [
        {"name": "Qwen/Qwen2.5-7B-Instruct", "family": "causal"},
        {"name": "meta-llama/Meta-Llama-3-8B-Instruct", "family": "causal"},
        {"name": "mistralai/Mistral-7B-Instruct-v0.2", "family": "causal"},
        {"name": "google/gemma-2b-it", "family": "causal"},
        {"name": "google/gemma-7b-it", "family": "causal"},
    ]

    # =====================================================
    # 5. TEXT UTILS
    # =====================================================

    def normalize_str(s: str) -> str:
        return " ".join(s.strip().lower().split())


    def normalize_key(s: str) -> str:
        """
        Normalize field labels for evaluation only.

        This is intentionally formatting-tolerant but not semantic/fuzzy:
        - Unicode NFKC normalization
        - case-insensitive matching via casefold()
        - trim surrounding whitespace
        - ignore common label punctuation/decoration
        - treat slash, backslash, hyphen, underscore, and vertical bar as word separators
        - collapse repeated whitespace

        Examples that become equivalent:
          "DATE:" == "date"
          "• CASE NAME :" == "Case Name"
          "NO. OF STORES" == "no of stores"
          "P.O.S." == "POS"
          "SENDER /PHONE NUMBER:" == "sender/phone number"

        No synonym expansion, fuzzy matching, stemming, or answer repair is used.
        """
        import unicodedata

        s = unicodedata.normalize("NFKC", str(s)).casefold().strip()
        if not s:
            return ""

        # Drop purely decorative leading/trailing characters.
        while s and not s[0].isalnum():
            s = s[1:]
        while s and not s[-1].isalnum():
            s = s[:-1]

        if not s:
            return ""

        out = []
        separators = {"/", "\\", "-", "_", "|"}
        removable_punct = {
            ".", ",", ":", ";", "'", '"', "`",
            "(", ")", "[", "]", "{", "}", "<", ">",
            "!", "?", "*", "#", "~", "^"
        }

        for ch in s:
            if ch.isalnum():
                out.append(ch)
            elif ch.isspace() or ch in separators:
                out.append(" ")
            elif ch in removable_punct:
                # Formatting punctuation inside labels is ignored.
                continue
            else:
                # Preserve non-formatting symbols (e.g., &, +, $) as tokens
                # so genuinely different labels are not silently merged.
                out.append(f" {ch} ")

        return " ".join("".join(out).split())

    def text_f1(gt: str, pred: str) -> float:
        """Token-level F1 with multiplicities (Counter overlap), not set overlap."""
        gt_tokens = normalize_str(gt).split()
        pred_tokens = normalize_str(pred).split()
        if not gt_tokens and not pred_tokens:
            return 1.0
        if not gt_tokens or not pred_tokens:
            return 0.0

        gt_c = Counter(gt_tokens)
        pred_c = Counter(pred_tokens)
        overlap = sum((gt_c & pred_c).values())
        if overlap == 0:
            return 0.0
        precision = overlap / len(pred_tokens)
        recall = overlap / len(gt_tokens)
        return 2 * precision * recall / (precision + recall)

    # =====================================================
    # 6. LOAD FUNSD KVP GROUND TRUTH
    # =====================================================

    def kvp_from_funsd_form(data: Dict[str, Any]) -> List[Dict[str, str]]:
        form = data.get("form", [])
        id2ent = {ent["id"]: ent for ent in form}

        kvps = []
        for ent in form:
            if ent.get("label", "").lower() != "question":
                continue

            qid = ent["id"]
            key_text = ent.get("text", "").strip()
            if not key_text:
                continue

            for link in ent.get("linking", []):
                if not isinstance(link, list) or len(link) != 2:
                    continue
                q_link, a_link = link
                if q_link != qid:
                    continue

                ans_ent = id2ent.get(a_link)
                if ans_ent is None:
                    continue
                val_text = ans_ent.get("text", "").strip()
                if not val_text:
                    continue

                kvps.append({"key": key_text, "value": val_text})
        return kvps

    def load_funsd_gt_kvp(annotation_dir: str) -> Dict[str, List[Dict[str, str]]]:
        gt = {}
        files = sorted(f for f in os.listdir(annotation_dir) if f.endswith(".json"))
        print(f"Found {len(files)} annotation files in {annotation_dir}")

        for fname in files:
            doc_id = os.path.splitext(fname)[0]
            fpath = os.path.join(annotation_dir, fname)
            with open(fpath, "r", encoding="utf-8") as f:
                data = json.load(f)

            if isinstance(data, list) and data and isinstance(data[0], dict) and "key" in data[0]:
                kvps = [{"key": str(d["key"]), "value": str(d.get("value", ""))} for d in data]
            elif isinstance(data, dict) and "form" in data:
                kvps = kvp_from_funsd_form(data)
            else:
                print(f"[WARN] Unknown annotation format for {fname}, skipping.")
                continue

            gt[doc_id] = kvps

        print(f"Loaded KVP ground truth for {len(gt)} documents.")
        return gt

    # =====================================================
    # 7. LOAD TEXT FILES (GOLD OR OCR)
    # =====================================================

    def load_doc_texts(text_dir: str) -> Dict[str, str]:
        texts = {}
        files = sorted(f for f in os.listdir(text_dir) if f.endswith(".txt"))
        print(f"Found {len(files)} text files in {text_dir}")

        for fname in files:
            doc_id = os.path.splitext(fname)[0]
            fpath = os.path.join(text_dir, fname)
            with open(fpath, "r", encoding="utf-8") as f:
                doc_text = f.read()
            texts[doc_id] = doc_text

        return texts

    # =====================================================
    # 8. PROMPT (ANTI-ANCHOR HARD SEPARATION)
    # =====================================================

    SYSTEM_MESSAGE = (
        "You extract key–value pairs (KVP) from OCR text of business documents. "
        "Return ONLY a JSON object with the exact schema {\"kvp\": [{\"key\":..., \"value\":...}, ...]}. "
        "Do not output any extra text."
    )

    # ---- BASE instructions used for ALL shots (including 0-shot)
    USER_INSTRUCTIONS_BASE = """
    Task: Extract ALL key–value pairs from the given document OCR text.

    Definition:
    - key = field label as written (keep exact casing/punctuation; do NOT rename or normalize).
    - value = the associated content following that key (same line or subsequent lines).

    Output rules (strict):
    - Output ONLY valid JSON (no markdown/code fences, no commentary).
    - JSON must be exactly: {"kvp": [{"key": "...", "value": "..."}, ...]}
    - Include every reasonable key–value pair you can extract.
    - If nothing is extractable: {"kvp": []}
    """.strip()

    # ---- FEW-SHOT ONLY anti-anchoring instructions (do NOT use in 0-shot)
    USER_INSTRUCTIONS_FEWSHOT = (USER_INSTRUCTIONS_BASE + """

    CRITICAL anti-anchoring rules:
    - Examples are ONLY to demonstrate the output format.
    - NEVER copy example keys or values.
    - Extract ONLY from <NEW_DOCUMENT>. Ignore all text inside any <EXAMPLE_*> blocks.
    """).strip()

    EXAMPLE_FILES_DIR = "/content/drive/MyDrive/data/funsd/funsd_prompt_examples"

    EX1_TXT  = os.path.join(EXAMPLE_FILES_DIR, "91391310.txt")
    EX1_JSON = os.path.join(EXAMPLE_FILES_DIR, "91391310_kvp.json")

    EX2_TXT  = os.path.join(EXAMPLE_FILES_DIR, "88547278_88547279.txt")
    # Historical filename; audited contents match annotation 88547278_88547279 (16 pairs).
    EX2_JSON = os.path.join(EXAMPLE_FILES_DIR, "89368010_kvp.json")

    EX3_TXT  = os.path.join(EXAMPLE_FILES_DIR, "0060165115.txt")
    EX3_JSON = os.path.join(EXAMPLE_FILES_DIR, "0060165115_kvp.json")

    def _read_text(path: str) -> str:
        if not os.path.exists(path):
            raise FileNotFoundError(
                f"Missing example file: {path}\n"
                f"Put your 3 example txt+json under: {EXAMPLE_FILES_DIR}"
            )
        with open(path, "r", encoding="utf-8") as f:
            return f.read().strip()

    def _read_kvp_json_as_prompt_obj(path: str) -> str:
        raw = _read_text(path)
        data = json.loads(raw)
        if isinstance(data, dict) and "kvp" in data:
            obj = data
        elif isinstance(data, list):
            obj = {"kvp": data}
        else:
            raise ValueError(f"Unexpected example KVP format in {path}: expected list or {{'kvp':[...]}}")
        return json.dumps(obj, ensure_ascii=False, indent=2)

    def load_funsd_examples() -> Dict[str, Dict[str, str]]:
        ex = {}
        ex["ex1"] = {"doc": _read_text(EX1_TXT), "json": _read_kvp_json_as_prompt_obj(EX1_JSON)}
        ex["ex2"] = {"doc": _read_text(EX2_TXT), "json": _read_kvp_json_as_prompt_obj(EX2_JSON)}
        ex["ex3"] = {"doc": _read_text(EX3_TXT), "json": _read_kvp_json_as_prompt_obj(EX3_JSON)}
        return ex

    # Only load examples when needed (1/2/3-shot).
    # This avoids failures if you run 0-shot but haven't staged example files.
    FUNSD_EX: Optional[Dict[str, Dict[str, str]]] = None

    def _ensure_examples_loaded():
        nonlocal FUNSD_EX
        if FUNSD_EX is None:
            FUNSD_EX = load_funsd_examples()

    def _example_block(idx: int, doc_text: str, out_json: str) -> str:
        return (
            f"<EXAMPLE_{idx}_DOCUMENT>\n{doc_text}\n</EXAMPLE_{idx}_DOCUMENT>\n"
            f"<EXAMPLE_{idx}_OUTPUT>\n{out_json}\n</EXAMPLE_{idx}_OUTPUT>\n"
        )

    def _task_footer(doc_text: str) -> str:
        return (
            "Now do the task for the NEW document below.\n"
            "Extract ONLY from <NEW_DOCUMENT>. Ignore all <EXAMPLE_*> blocks completely.\n"
            "Return ONLY a SINGLE JSON object and stop immediately after the final '}' character.\n"
            "<NEW_DOCUMENT>\n"
            f"{doc_text}\n"
            "</NEW_DOCUMENT>\n"
            "Output:\n"
        )

    def build_prompt_0shot(doc_text: str) -> str:
        prompt = USER_INSTRUCTIONS_BASE + "\n\n" + _task_footer(doc_text)
        # Guard only against real example blocks (not text mentioning "<EXAMPLE_*>")
        assert "<EXAMPLE_1_DOCUMENT>" not in prompt, "0-shot prompt contains example blocks unexpectedly."
        return prompt

    def build_prompt_1shot(doc_text: str) -> str:
        _ensure_examples_loaded()
        return "\n\n".join([
            USER_INSTRUCTIONS_FEWSHOT,
            _example_block(1, FUNSD_EX["ex1"]["doc"], FUNSD_EX["ex1"]["json"]),
            _task_footer(doc_text),
        ])

    def build_prompt_2shot(doc_text: str) -> str:
        _ensure_examples_loaded()
        return "\n\n".join([
            USER_INSTRUCTIONS_FEWSHOT,
            _example_block(1, FUNSD_EX["ex1"]["doc"], FUNSD_EX["ex1"]["json"]),
            _example_block(2, FUNSD_EX["ex2"]["doc"], FUNSD_EX["ex2"]["json"]),
            _task_footer(doc_text),
        ])

    def build_prompt_3shot(doc_text: str) -> str:
        _ensure_examples_loaded()
        return "\n\n".join([
            USER_INSTRUCTIONS_FEWSHOT,
            _example_block(1, FUNSD_EX["ex1"]["doc"], FUNSD_EX["ex1"]["json"]),
            _example_block(2, FUNSD_EX["ex2"]["doc"], FUNSD_EX["ex2"]["json"]),
            _example_block(3, FUNSD_EX["ex3"]["doc"], FUNSD_EX["ex3"]["json"]),
            _task_footer(doc_text),
        ])

    def build_prompt(doc_text: str) -> str:
        if SHOT_VARIANT == "0shot":
            return build_prompt_0shot(doc_text)
        if SHOT_VARIANT == "1shot":
            return build_prompt_1shot(doc_text)
        if SHOT_VARIANT == "2shot":
            return build_prompt_2shot(doc_text)
        if SHOT_VARIANT == "3shot":
            return build_prompt_3shot(doc_text)
        raise ValueError(f"Unknown SHOT_VARIANT: {SHOT_VARIANT}")

    # =====================================================
    # 9. PARSING
    # =====================================================

    def parse_kvp_output(raw_output: str) -> List[Dict[str, str]]:
        """
        Parse LLM output conservatively while preserving complete KVP objects.

        1. Prefer a complete valid JSON response:
           {"kvp": [{"key": "...", "value": "..."}, ...]}
        2. If the full response is malformed/truncated, recover only individually
           COMPLETE, valid JSON objects already present in the generated text
           that contain string-valued "key" and "value" fields.

        No regex KVP extraction, malformed-JSON repair, guessing, fuzzy parsing,
        GT-aware recovery, or document-text fallback is performed.
        """
        if not isinstance(raw_output, str):
            return []

        s = raw_output.strip()
        if not s:
            return []

        if s.startswith("```"):
            lines = s.splitlines()
            if lines and lines[0].strip().startswith("```"):
                lines = lines[1:]
            if lines and lines[-1].strip() == "```":
                lines = lines[:-1]
            s = "\n".join(lines).strip()

        try:
            obj = json.loads(s)
        except json.JSONDecodeError:
            obj = None

        if isinstance(obj, dict):
            kvp_list = obj.get("kvp")
            if isinstance(kvp_list, list):
                parsed = []
                for pair in kvp_list:
                    if not isinstance(pair, dict):
                        continue
                    key = pair.get("key")
                    value = pair.get("value")
                    if not isinstance(key, str) or not isinstance(value, str):
                        continue
                    key = key.strip()
                    value = value.strip()
                    if key == "" and value == "":
                        continue
                    parsed.append({"key": key, "value": value})
                return parsed

        decoder = json.JSONDecoder()
        recovered = []

        for start, char in enumerate(s):
            if char != "{":
                continue
            try:
                candidate, _ = decoder.raw_decode(s[start:])
            except json.JSONDecodeError:
                continue

            if not isinstance(candidate, dict):
                continue

            key = candidate.get("key")
            value = candidate.get("value")
            if not isinstance(key, str) or not isinstance(value, str):
                continue

            key = key.strip()
            value = value.strip()
            if key == "" and value == "":
                continue

            recovered.append({"key": key, "value": value})

        return recovered

    # =====================================================
    # 10. METRICS
    # =====================================================

    # NOTE ON METRICS:
    # - Key Recall is duplicate-aware and uses formatting-tolerant normalized keys.
    # - Exact Match requires the normalized key and normalized value to match exactly.
    # - Value F1 is token-overlap F1 on values, matched by normalized key with no prediction reuse.
    #   Every GT pair contributes to the final average; missing keys and zero-overlap values receive 0.
    #   This implementation is intentionally aligned with the manuscript definition.

    def compute_doc_metrics(
        gt_kvp: List[Dict[str, str]],
        pred_kvp: List[Dict[str, str]]
    ) -> Dict[str, Any]:

        num_gt = len(gt_kvp)

        if num_gt == 0:
            return {
                "num_gt_keys": 0,
                "matched_keys": 0,
                "key_recall": 0.0,
                "exact_matches": 0,
                "exact_match_rate": 0.0,
                "value_f1": 0.0,
            }

        # -------------------------------------------------
        # Normalize GT and predicted key/value strings
        # -------------------------------------------------
        gt_keys_norm = [
            normalize_key(kv.get("key", ""))
            for kv in gt_kvp
        ]
        gt_vals_norm = [
            normalize_str(kv.get("value", ""))
            for kv in gt_kvp
        ]

        pred_keys_norm = [
            normalize_key(kv.get("key", ""))
            for kv in pred_kvp
        ]
        pred_vals_norm = [
            normalize_str(kv.get("value", ""))
            for kv in pred_kvp
        ]

        # =================================================
        # 1. KEY RECALL
        # Duplicate-aware key matching
        # =================================================

        gt_key_counts = Counter(k for k in gt_keys_norm if k)
        pred_key_counts = Counter(k for k in pred_keys_norm if k)

        matched_keys = sum(
            min(gt_key_counts[k], pred_key_counts.get(k, 0))
            for k in gt_key_counts
        )

        key_recall = matched_keys / num_gt

        # =================================================
        # 2. EXACT MATCH RATE
        # Exact normalized key AND exact normalized value.
        # Each prediction can be used at most once.
        # =================================================

        used_pred_exact = set()
        exact_matches = 0

        for gt_key, gt_value in zip(gt_keys_norm, gt_vals_norm):

            if not gt_key or not gt_value:
                continue

            for j, (pred_key, pred_value) in enumerate(
                zip(pred_keys_norm, pred_vals_norm)
            ):
                if j in used_pred_exact:
                    continue

                if gt_key == pred_key and gt_value == pred_value:
                    exact_matches += 1
                    used_pred_exact.add(j)
                    break

        exact_match_rate = exact_matches / num_gt

        # =================================================
        # 3. VALUE TOKEN F1
        #
        # For every GT pair:
        #   - consider unused predictions with the same key
        #   - select the one with maximum token-level F1
        #   - same-key but zero-overlap value -> F1 = 0
        #   - no same-key prediction -> F1 = 0
        #
        # Final Value F1 is averaged over ALL GT pairs.
        # =================================================

        used_pred_f1 = set()
        value_f1_scores = []

        for gt_pair in gt_kvp:

            gt_key = normalize_key(gt_pair.get("key", ""))
            gt_value = str(gt_pair.get("value", ""))

            # A malformed/empty GT field receives zero credit.
            if not gt_key or not normalize_str(gt_value):
                value_f1_scores.append(0.0)
                continue

            best_j = None
            best_f1 = 0.0

            for j, pred_pair in enumerate(pred_kvp):

                if j in used_pred_f1:
                    continue

                pred_key = normalize_key(pred_pair.get("key", ""))

                if pred_key != gt_key or not pred_key:
                    continue

                pred_value = str(pred_pair.get("value", ""))

                f1 = text_f1(gt_value, pred_value)

                # Important:
                # best_j must also be assigned when F1 == 0.
                if best_j is None or f1 > best_f1:
                    best_j = j
                    best_f1 = f1

            if best_j is not None:
                used_pred_f1.add(best_j)
                value_f1_scores.append(best_f1)
            else:
                # Missing key
                value_f1_scores.append(0.0)

        value_f1 = float(np.mean(value_f1_scores))

        return {
            "num_gt_keys": num_gt,
            "matched_keys": int(matched_keys),
            "key_recall": float(key_recall),
            "exact_matches": int(exact_matches),
            "exact_match_rate": float(exact_match_rate),
            "value_f1": float(value_f1),
        }
    # =====================================================
    # 11. MODEL LOADING + GENERATION (NO BNB)
    # =====================================================

    def _preferred_dtype() -> torch.dtype:
        if DEVICE == "cuda":
            return torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
        return torch.float32

    def build_model_and_tokenizer(model_name: str, family: str):
        print(f"Loading model: {model_name}  (family={family})")

        # Proactively clear cache before loading a big model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()

        kwargs = {}
        if HF_TOKEN:
            kwargs["token"] = HF_TOKEN

        config = AutoConfig.from_pretrained(model_name, **kwargs)
        tokenizer = AutoTokenizer.from_pretrained(model_name, **kwargs)

        is_encoder_decoder = bool(getattr(config, "is_encoder_decoder", False)) or (family == "seq2seq")

        if not is_encoder_decoder:
            tokenizer.padding_side = "left"

        dtype = _preferred_dtype()

        if DEVICE == "cuda":
            primary_device_map = "cuda"
        else:
            primary_device_map = None

        def _load(device_map):
            if is_encoder_decoder:
                return AutoModelForSeq2SeqLM.from_pretrained(
                    model_name,
                    device_map=device_map,
                    dtype=dtype,
                    **kwargs,
                )
            return AutoModelForCausalLM.from_pretrained(
                model_name,
                device_map=device_map,
                dtype=dtype,
                **kwargs,
            )

        try:
            model = _load(primary_device_map)
        except Exception as e:
            # Do not silently change device placement. A failed CUDA load must be
            # surfaced so the run cannot continue under a different hardware
            # configuration (for example CPU offloading via device_map="auto").
            raise RuntimeError(
                f"Failed to load {model_name} with device_map={primary_device_map!r}. "
                "Check local disk space/model cache and GPU memory before retrying."
            ) from e

        if tokenizer.pad_token is None and tokenizer.eos_token is not None:
            tokenizer.pad_token = tokenizer.eos_token
            model.config.pad_token_id = tokenizer.eos_token_id

        print("hf_device_map:", getattr(model, "hf_device_map", None))
        model.eval()
        return tokenizer, model, is_encoder_decoder

    def _effective_max_input_tokens(tokenizer) -> int:
        mml = getattr(tokenizer, "model_max_length", None)
        if mml is None or mml > 100000:
            return MAX_INPUT_TOKENS
        return int(min(MAX_INPUT_TOKENS, mml))

    def _maybe_apply_chat_template(tokenizer, user_prompt: str) -> str:
        try:
            tmpl = getattr(tokenizer, "chat_template", None)
            if tmpl:
                messages = [
                    {"role": "system", "content": SYSTEM_MESSAGE},
                    {"role": "user", "content": user_prompt},
                ]
                return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        except Exception:
            pass
        return user_prompt

    def _qwen_im_end_eos_id(tokenizer) -> Optional[int]:
        try:
            im_end_id = tokenizer.convert_tokens_to_ids("<|im_end|>")
            if im_end_id is not None and im_end_id != tokenizer.unk_token_id:
                return int(im_end_id)
        except Exception:
            pass
        return None

    def _postprocess_decoded_text(s: str) -> str:
        s = s.strip()
        s = s.replace("<|im_end|>", "").strip()
        return s

    def run_model_on_prompts(tokenizer, model, prompts: List[str], is_encoder_decoder: bool):
        """Generate outputs plus neutral generation-length diagnostics."""
        if not is_encoder_decoder:
            prompts = [_maybe_apply_chat_template(tokenizer, p) for p in prompts]

        max_in = _effective_max_input_tokens(tokenizer)

        inputs = tokenizer(
            prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=max_in,
        )

        if DEVICE == "cuda":
            inputs = {k: v.to("cuda") for k, v in inputs.items()}

        eos_id = _qwen_im_end_eos_id(tokenizer) or int(tokenizer.eos_token_id)

        gen_kwargs = dict(
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=int(tokenizer.pad_token_id),
            eos_token_id=int(eos_id),
            return_dict_in_generate=True,
            output_scores=False,
        )

        with torch.no_grad():
            gen_out = model.generate(**inputs, **gen_kwargs)

        sequences = gen_out.sequences
        results = []

        if is_encoder_decoder:
            for i in range(sequences.shape[0]):
                ids = sequences[i]
                generated_tokens = int(ids.shape[-1])
                text = tokenizer.decode(ids, skip_special_tokens=True).strip()
                results.append({
                    "text": text,
                    "generated_tokens": generated_tokens,
                    "output_hit_max_new_tokens": bool(generated_tokens >= MAX_NEW_TOKENS),
                })
            return results

        input_width = int(inputs["input_ids"].shape[1])

        for i in range(sequences.shape[0]):
            ids = sequences[i, input_width:]
            generated_tokens = int(ids.shape[-1])
            text = tokenizer.decode(ids, skip_special_tokens=False)
            text = _postprocess_decoded_text(text)

            results.append({
                "text": text,
                "generated_tokens": generated_tokens,
                "output_hit_max_new_tokens": bool(generated_tokens >= MAX_NEW_TOKENS),
            })

        return results

    # =====================================================
    # 12. PROMPT TOKEN LOGGING
    # =====================================================

    def prompt_token_stats(tokenizer, prompt: str) -> Tuple[int, int, bool]:
        max_in = _effective_max_input_tokens(tokenizer)
        tok_no_trunc = tokenizer(prompt, return_tensors="pt", truncation=False)
        tok_trunc    = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=max_in)
        n_no_trunc = int(tok_no_trunc["input_ids"].shape[-1])
        n_trunc    = int(tok_trunc["input_ids"].shape[-1])
        return n_no_trunc, n_trunc, (n_no_trunc > n_trunc)

    # =====================================================
    # 13. MAIN EVAL LOOP
    # =====================================================

    def _cleanup_cuda():
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()

    def evaluate_model_on_funsd(model_name: str, family: str):
        tokenizer = None
        model = None
        is_encoder_decoder = False

        try:
            print("\n" + "=" * 60)
            print(f"=== Running model {model_name} on FUNSD ({ENGINE}, {SHOT_VARIANT}) ===")

            gt_dict   = load_funsd_gt_kvp(ANNOTATION_DIR)
            text_dict = load_doc_texts(TEXT_DIR)

            all_common_doc_ids = sorted(set(gt_dict.keys()) & set(text_dict.keys()))

            # KVP metrics are undefined for documents with no annotated
            # ground-truth KVP targets. Exclude those documents systematically
            # for every FUNSD condition (Gold, Tesseract, EasyOCR, PaddleOCR).
            zero_gt_doc_ids = [
                d for d in all_common_doc_ids
                if len(gt_dict.get(d, [])) == 0
            ]
            doc_ids = [
                d for d in all_common_doc_ids
                if len(gt_dict.get(d, [])) > 0
            ]

            if MAX_DOCS is not None:
                doc_ids = doc_ids[:MAX_DOCS]

            print(f"FUNSD docs with both GT and text: {len(all_common_doc_ids)}")
            print(f"Excluded zero-GT KVP docs: {len(zero_gt_doc_ids)}")
            if zero_gt_doc_ids:
                print("  Excluded IDs:", zero_gt_doc_ids)
            print(f"Evaluable FUNSD docs: {len(doc_ids)}")

            out_dir_name = f"funsd_{ENGINE}_{SHOT_VARIANT}_{model_name.replace('/', '_')}"
            OUTPUT_DIR   = os.path.join(RESULTS_BASE, out_dir_name)
            os.makedirs(OUTPUT_DIR, exist_ok=True)
            print(f"Results will be saved under: {OUTPUT_DIR}")

            csv_path  = os.path.join(OUTPUT_DIR, f"metrics_{ENGINE}_{SHOT_VARIANT}_{model_name.replace('/', '_')}.csv")
            pred_path = os.path.join(OUTPUT_DIR, f"predictions_{ENGINE}_{SHOT_VARIANT}_{model_name.replace('/', '_')}.jsonl")

            if not RESUME:
                if os.path.exists(csv_path):
                    os.remove(csv_path)
                if os.path.exists(pred_path):
                    os.remove(pred_path)

            processed_ids = set()
            csv_exists = os.path.exists(csv_path)
            if RESUME and csv_exists:
                with open(csv_path, "r", encoding="utf-8") as f:
                    lines = f.readlines()
                if len(lines) > 1:
                    for line in lines[1:]:
                        parts = line.strip().split(",")
                        if parts and parts[0]:
                            processed_ids.add(parts[0])
                eligible_processed_ids = processed_ids.intersection(set(doc_ids))
                print(
                    f"Found existing metrics CSV with {len(processed_ids)} rows; "
                    f"{len(eligible_processed_ids)} belong to the current evaluable FUNSD subset."
                )
                processed_ids = eligible_processed_ids

            doc_ids_to_run = [d for d in doc_ids if d not in processed_ids]
            print(f"Docs remaining for this model: {len(doc_ids_to_run)}")

            if len(doc_ids_to_run) != 0:
                try:
                    tokenizer, model, is_encoder_decoder = build_model_and_tokenizer(model_name, family)
                except Exception as e:
                    print(f"[ERROR] Could not load model {model_name}: {e}")
                    print("Skipping this model.\n")
                    return

                csv_mode = "a" if (RESUME and csv_exists) else "w"
                with open(csv_path, csv_mode, encoding="utf-8") as csv_f, \
                     open(pred_path, "a", encoding="utf-8") as pred_f:

                    if csv_mode == "w":
                        csv_f.write("doc_id,num_gt_keys,matched_keys,key_recall,exact_match_rate,value_f1,prompt_tokens_no_trunc,prompt_tokens_trunc,prompt_truncated,generated_tokens,output_hit_max_new_tokens\n")

                    with tqdm(
                        total=len(doc_ids_to_run),
                        desc=f"{model_name} [{ENGINE}][{SHOT_VARIANT}]",
                        dynamic_ncols=True
                    ) as pbar:

                        for start in range(0, len(doc_ids_to_run), BATCH_SIZE):
                            batch_ids    = doc_ids_to_run[start:start + BATCH_SIZE]
                            batch_texts  = [text_dict[d] for d in batch_ids]
                            batch_gt_kvp = [gt_dict[d] for d in batch_ids]

                            batch_prompts = [build_prompt(t) for t in batch_texts]
                            batch_prompt_stats = [prompt_token_stats(tokenizer, p) for p in batch_prompts]

                            if DEBUG_FIRST_DOC and start == 0:
                                p0 = batch_prompts[0]
                                max_in = _effective_max_input_tokens(tokenizer)
                                n_no_trunc, n_trunc, truncated = batch_prompt_stats[0]
                                print("\n===== DEBUG: PROMPT TOKEN ANALYSIS =====")
                                print(f"Tokenizer model_max_length: {getattr(tokenizer, 'model_max_length', None)}")
                                print(f"Effective max_input_tokens: {max_in}")
                                print(f"Prompt tokens (no trunc):   {n_no_trunc}")
                                print(f"Prompt tokens (truncated):  {n_trunc}")
                                if truncated:
                                    print("⚠️ Prompt was truncated.")
                                print("Contains <NEW_DOCUMENT> footer:", "<NEW_DOCUMENT>" in p0 and "</NEW_DOCUMENT>" in p0)
                                print("\n----- PROMPT TAIL (last 1200 chars) -----")
                                print(p0[-1200:])
                                print("===== END DEBUG =====\n")

                            generation_results = run_model_on_prompts(tokenizer, model, batch_prompts, is_encoder_decoder)

                            for (doc_id, doc_text, gt_kvp, gen_result, pstats) in zip(
                                batch_ids, batch_texts, batch_gt_kvp, generation_results, batch_prompt_stats
                            ):
                                n_no_trunc, n_trunc, truncated = pstats
                                raw_output = gen_result["text"]
                                generated_tokens = int(gen_result["generated_tokens"])
                                output_hit_max_new_tokens = bool(gen_result["output_hit_max_new_tokens"])

                                pred_kvp = parse_kvp_output(raw_output)
                                metrics  = compute_doc_metrics(gt_kvp, pred_kvp)

                                csv_f.write(
                                    f"{doc_id},{metrics['num_gt_keys']},{metrics['matched_keys']},"
                                    f"{metrics['key_recall']:.10f},{metrics['exact_match_rate']:.10f},"
                                    f"{metrics['value_f1']:.10f},{n_no_trunc},{n_trunc},{int(truncated)},"
                                    f"{generated_tokens},{int(output_hit_max_new_tokens)}\n"
                                )
                                csv_f.flush()
                                os.fsync(csv_f.fileno())

                                record = {
                                    "doc_id": doc_id,
                                    "engine": ENGINE,
                                    "shot_variant": SHOT_VARIANT,
                                    "model": model_name,
                                    "gt_kvp": gt_kvp,
                                    "pred_kvp": pred_kvp,
                                    "raw_output": raw_output,
                                    "metrics": metrics,
                                    "prompt_tokens_no_trunc": n_no_trunc,
                                    "prompt_tokens_trunc": n_trunc,
                                    "prompt_truncated": bool(truncated),
                                    "generated_tokens": generated_tokens,
                                    "output_hit_max_new_tokens": output_hit_max_new_tokens,
                                    "seed": SEED,
                                    "dtype": str(_preferred_dtype()),
                                }
                                pred_f.write(json.dumps(record) + "\n")
                                pred_f.flush()
                                os.fsync(pred_f.fileno())

                                pbar.update(1)

            # ----- Final summary from CSV -----
            macro_key_recall = 0.0
            macro_em         = 0.0
            macro_value_f1   = 0.0
            n_docs           = 0

            if os.path.exists(csv_path):
                with open(csv_path, "r", encoding="utf-8") as f:
                    lines = f.readlines()
                if len(lines) > 1:
                    sum_key_rec = 0.0
                    sum_em      = 0.0
                    sum_f1      = 0.0
                    for line in lines[1:]:
                        parts = line.strip().split(",")
                        if len(parts) < 6:
                            continue
                        try:
                            num_gt  = int(parts[1])
                            key_rec = float(parts[3])
                            em      = float(parts[4])
                            f1      = float(parts[5])
                        except ValueError:
                            continue

                        # Defensive compatibility with older FUNSD CSVs:
                        # zero-GT documents are not part of KVP macro evaluation.
                        if num_gt == 0:
                            continue

                        sum_key_rec += key_rec
                        sum_em      += em
                        sum_f1      += f1
                        n_docs      += 1
                    if n_docs > 0:
                        macro_key_recall = sum_key_rec / n_docs
                        macro_em         = sum_em / n_docs
                        macro_value_f1   = sum_f1 / n_docs

            print("\n==== SUMMARY (MACRO AVERAGE OVER DOCS) ====")
            print(f"Dataset: funsd")
            print(f"Engine:  {ENGINE}")
            print(f"Shot:    {SHOT_VARIANT}")
            print(f"Model:   {model_name}")
            print(f"Evaluable docs in macro average: {n_docs}")
            print(f"Key Recall: {macro_key_recall:.4f}")
            print(f"EM:         {macro_em:.4f}")
            print(f"Value F1:   {macro_value_f1:.4f}")
            print(f"\nPer-doc metrics saved to: {csv_path}")
            print(f"Predictions saved to:      {pred_path}")
            print("=" * 60 + "\n")

        finally:
            # Always cleanup GPU/CPU memory between runs
            try:
                if model is not None:
                    del model
                if tokenizer is not None:
                    del tokenizer
            except Exception:
                pass
            _cleanup_cuda()

    # =====================================================
    # 14. MAIN
    # =====================================================

    def main():
        for spec in tqdm(MODEL_SPECS, desc=f"Models [{SHOT_VARIANT}]", dynamic_ncols=True):
            evaluate_model_on_funsd(
                model_name=spec["name"],
                family=spec["family"],
            )

    # The runner is intentionally not auto-executed here.
    # Use the experiment cells later in the notebook.

    def configure(engine, shot, model_specs):
        """Configure one FUNSD experiment without modifying any other dataset."""
        nonlocal ENGINE, SHOT_VARIANT, TEXT_DIR, MODEL_SPECS, RESUME, RESULTS_BASE

        valid_engines = {"gold_text_spatial", "gold_text", "tesseract", "paddleocr", "easyocr"}
        valid_shots = {"0shot", "1shot", "2shot", "3shot"}

        if engine not in valid_engines:
            raise ValueError(f"Unsupported engine: {engine}")
        if shot not in valid_shots:
            raise ValueError(f"Unsupported shot setting: {shot}")

        ENGINE = engine
        SHOT_VARIANT = shot
        MODEL_SPECS = list(model_specs)
        RESUME = True
        RESULTS_BASE = "/content/drive/MyDrive/data/pixels_to_pairs_final_results/funsd"
        os.makedirs(RESULTS_BASE, exist_ok=True)

        text_paths = {
            "gold_text_spatial": os.path.join(BASE_DIR, "funsd_gold_text_spatial"),
            "gold_text": os.path.join(BASE_DIR, "funsd_gold_text"),
            "tesseract": os.path.join(BASE_DIR, "funsd_ocr", "tesseract"),
            "paddleocr": os.path.join(BASE_DIR, "funsd_ocr", "paddleocr"),
            "easyocr": os.path.join(BASE_DIR, "funsd_ocr", "easyocr"),
        }
        TEXT_DIR = text_paths[engine]

        if "/pixels_to_pairs_final_results/funsd" not in RESULTS_BASE:
            raise RuntimeError(f"FUNSD results path is incorrect: {RESULTS_BASE}")
        if "/data/funsd/" not in TEXT_DIR:
            raise RuntimeError(f"FUNSD text path is incorrect: {TEXT_DIR}")

        print("\nCONFIG CHECK — FUNSD")
        print("  ENGINE:", ENGINE)
        print("  SHOT:", SHOT_VARIANT)
        print("  TEXT_DIR:", TEXT_DIR)
        print("  RESULTS_BASE:", RESULTS_BASE)
        print("  MAX_NEW_TOKENS:", MAX_NEW_TOKENS)
        print("  MODELS:", [m["name"] for m in MODEL_SPECS])

    def run(engine, shot, model_specs):
        """Run one complete FUNSD condition sequentially."""
        configure(engine, shot, model_specs)
        for spec in MODEL_SPECS:
            evaluate_model_on_funsd(spec["name"], spec["family"])

    def status():
        return {
            "dataset": "FUNSD",
            "engine": ENGINE,
            "shot": SHOT_VARIANT,
            "text_dir": TEXT_DIR,
            "results_base": RESULTS_BASE,
            "max_new_tokens": MAX_NEW_TOKENS,
            "resume": RESUME,
        }

    return SimpleNamespace(
        configure=configure,
        run=run,
        status=status,
        evaluate=evaluate_model_on_funsd,
    )

FUNSD_RUNNER = build_funsd_runner()
print("FUNSD runner ready:", FUNSD_RUNNER.status())


### FUNSD normalization-collision audit

Run the next cell once before the full benchmark. It checks whether the formatting-tolerant key normalizer maps **different raw FUNSD GT labels** to the same normalized key. The important result is the **within-document collision count**. A global collision can be harmless when the labels occur in different documents; a within-document collision can make matching ambiguous and should be reviewed.

This cell is diagnostic only and does **not** change GT, predictions, or metrics.


In [ ]:
# ============================================================
# FUNSD KEY-NORMALIZATION COLLISION AUDIT
# ============================================================
# Purpose:
# Verify that the formatting-tolerant key normalizer does not
# collapse genuinely different FUNSD GT labels into the same key.
#
# This is diagnostic only. It does NOT modify predictions, GT,
# metrics, or the experiment subset.

from collections import defaultdict
import json
import os
import unicodedata

def _audit_normalize_key(s: str) -> str:
    """
    Exact copy of the benchmark's formatting-tolerant key normalizer.
    Keep this synchronized with normalize_key() in the FUNSD runner.
    """
    s = unicodedata.normalize("NFKC", str(s)).casefold().strip()
    if not s:
        return ""

    while s and not s[0].isalnum():
        s = s[1:]
    while s and not s[-1].isalnum():
        s = s[:-1]

    if not s:
        return ""

    out = []
    separators = {"/", "\\", "-", "_", "|"}
    removable_punct = {
        ".", ",", ":", ";", "'", '"', "`",
        "(", ")", "[", "]", "{", "}", "<", ">",
        "!", "?", "*", "#", "~", "^"
    }

    for ch in s:
        if ch.isalnum():
            out.append(ch)
        elif ch.isspace() or ch in separators:
            out.append(" ")
        elif ch in removable_punct:
            continue
        else:
            out.append(f" {ch} ")

    return " ".join("".join(out).split())


def _extract_funsd_gt_pairs_from_annotation(annotation_path):
    """
    Reconstruct the same question->answer KVP targets from a raw FUNSD
    annotation file for collision auditing.

    Only links where one endpoint is labeled 'question' and the other
    is labeled 'answer' are treated as KVP targets. Header/question
    links and other relation types are ignored.
    """
    with open(annotation_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    items = data.get("form", [])
    by_id = {int(x["id"]): x for x in items if "id" in x}

    # FUNSD repeats links on both entities in some files, so de-duplicate.
    relations = set()
    for item in items:
        for link in item.get("linking", []) or []:
            if not isinstance(link, (list, tuple)) or len(link) != 2:
                continue
            try:
                a, b = int(link[0]), int(link[1])
            except Exception:
                continue
            relations.add(tuple(sorted((a, b))))

    pairs = []
    for a, b in relations:
        ia = by_id.get(a)
        ib = by_id.get(b)
        if not ia or not ib:
            continue

        la = str(ia.get("label", "")).lower()
        lb = str(ib.get("label", "")).lower()

        if la == "question" and lb == "answer":
            q, ans = ia, ib
        elif la == "answer" and lb == "question":
            q, ans = ib, ia
        else:
            continue

        q_text = str(q.get("text", "")).strip()
        a_text = str(ans.get("text", "")).strip()

        if q_text:
            pairs.append({"key": q_text, "value": a_text})

    return pairs


def audit_funsd_key_normalization_collisions(annotation_dir):
    """
    Report whether distinct raw GT labels collapse to the same normalized key.

    Two levels are reported:
      1. GLOBAL collisions: raw labels anywhere in the test set map to the
         same normalized form.
      2. WITHIN-DOCUMENT collisions: distinct raw labels in the SAME document
         map to the same normalized form. These are the important ones for
         possible evaluation ambiguity.

    Repeated occurrences of the exact same raw key (e.g. table columns such
    as 'NAME OF ACCOUNT') are not treated as a normalization collision.
    """
    if not os.path.isdir(annotation_dir):
        raise FileNotFoundError(f"FUNSD annotation directory not found: {annotation_dir}")

    files = sorted(
        f for f in os.listdir(annotation_dir)
        if f.lower().endswith(".json")
    )

    global_map = defaultdict(set)
    within_doc_collisions = []
    zero_gt_docs = []
    evaluable_docs = 0
    total_pairs = 0

    for filename in files:
        path = os.path.join(annotation_dir, filename)
        doc_id = os.path.splitext(filename)[0]
        pairs = _extract_funsd_gt_pairs_from_annotation(path)

        if len(pairs) == 0:
            zero_gt_docs.append(doc_id)
            continue

        evaluable_docs += 1
        total_pairs += len(pairs)

        local_map = defaultdict(set)

        for pair in pairs:
            raw_key = pair["key"]
            norm_key = _audit_normalize_key(raw_key)
            global_map[norm_key].add(raw_key)
            local_map[norm_key].add(raw_key)

        for norm_key, raw_keys in sorted(local_map.items()):
            if len(raw_keys) > 1:
                within_doc_collisions.append({
                    "doc_id": doc_id,
                    "normalized_key": norm_key,
                    "raw_keys": sorted(raw_keys),
                })

    global_collisions = [
        {
            "normalized_key": norm_key,
            "raw_keys": sorted(raw_keys),
        }
        for norm_key, raw_keys in sorted(global_map.items())
        if len(raw_keys) > 1
    ]

    print("=" * 72)
    print("FUNSD KEY-NORMALIZATION COLLISION AUDIT")
    print("=" * 72)
    print(f"Annotation files scanned:          {len(files)}")
    print(f"Evaluable docs (>0 GT KVPs):       {evaluable_docs}")
    print(f"Zero-GT docs excluded:             {len(zero_gt_docs)}")
    if zero_gt_docs:
        print(f"Zero-GT IDs:                       {zero_gt_docs}")
    print(f"Total GT KVP pairs audited:        {total_pairs}")
    print(f"Global normalized-key collisions:  {len(global_collisions)}")
    print(f"Within-document collisions:        {len(within_doc_collisions)}")
    print()

    if global_collisions:
        print("GLOBAL COLLISIONS")
        print("-" * 72)
        for item in global_collisions:
            print(f"{item['normalized_key']!r} <- {item['raw_keys']}")
        print()

    if within_doc_collisions:
        print("WITHIN-DOCUMENT COLLISIONS — REVIEW THESE")
        print("-" * 72)
        for item in within_doc_collisions:
            print(
                f"{item['doc_id']}: {item['normalized_key']!r} "
                f"<- {item['raw_keys']}"
            )
    else:
        print("PASS: No within-document collisions between distinct raw GT labels.")

    return {
        "annotation_files": len(files),
        "evaluable_docs": evaluable_docs,
        "zero_gt_docs": zero_gt_docs,
        "total_gt_pairs": total_pairs,
        "global_collisions": global_collisions,
        "within_doc_collisions": within_doc_collisions,
    }


# Update this path only if your raw FUNSD annotation folder is elsewhere.
FUNSD_ANNOTATION_DIR = "/content/drive/MyDrive/data/funsd/annotations"

funsd_collision_audit = audit_funsd_key_normalization_collisions(
    FUNSD_ANNOTATION_DIR
)


### Archived diagnostic removed from Run All

The old hard-coded FUNSD `gold_text` prediction inspection cell was intentionally disabled because those results are obsolete after the spatial-Gold correction.


### SROIE runner definition

In [ ]:
def build_sroie_runner():
    # =====================================================
    # Pixels to Pairs benchmark — SROIE
    # Final benchmark runner
    # =====================================================
    # This cell preserves the recovered experimental prompt/parsing/generation
    # logic for SROIE, while cleaning secrets/paths and using the corrected,
    # manuscript-aligned metric implementation.
    #
    # Before running:
    #   1) Mount Google Drive using the setup cell above.
    #   2) Set HF_TOKEN in the Colab environment if a gated model requires it.
    #   3) Check ENGINE, SHOT_VARIANT, MODEL_SPECS, and dataset paths.
    #   4) Use the experiment cells later in this notebook; they run configurations sequentially.
    #
    import os
    import json
    import gc
    from typing import Dict, List, Any, Optional, Tuple
    from collections import Counter

    import numpy as np
    import torch
    from transformers import (
        AutoTokenizer,
        AutoModelForSeq2SeqLM,
        AutoModelForCausalLM,
        AutoConfig,
    )
    from tqdm.auto import tqdm

    # =====================================================
    # 0. ENV (CUDA ALLOC)
    # =====================================================

    os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

    # =====================================================
    # 1. REPRODUCIBILITY (LIGHT)
    # =====================================================

    SEED = 0
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)

    # =====================================================
    # 2. HF TOKEN & PERSISTENT CACHE
    # =====================================================

    # Optional: put your HF token in the environment (or leave as None)
    HF_TOKEN = os.environ.get("HF_TOKEN")
    import os

    HF_CACHE_DIR = "/root/.cache/huggingface"
    os.makedirs(HF_CACHE_DIR, exist_ok=True)

    os.environ["HF_HOME"] = HF_CACHE_DIR
    os.environ["TRANSFORMERS_CACHE"] = HF_CACHE_DIR
    print(f"Using HF cache dir: {HF_CACHE_DIR}")

    # =====================================================
    # 3. CONFIG – PATHS & GLOBALS (SROIE)
    # =====================================================

    # This assumes you have:
    #   /content/drive/MyDrive/data/sroie/
    #       ├── annotations_test/   (or similar)  <-- JSON with KVP
    #       └── sroie_gold_text/    <-- gold OCR text (.txt)
    #
    # You can tweak KVP_DIR and TEXT_DIR as needed.

    BASE_DIR = "/content/drive/MyDrive/data/sroie"

    # Where the KVP JSON annotations live (e.g., test set)
    KVP_DIR = os.path.join(BASE_DIR, "sroie_kvp")

    # Which text engine to use
    ENGINE = "tesseract"  # "gold_text" | "tesseract" | "paddleocr" | "easyocr"

    if ENGINE == "gold_text":
        TEXT_DIR = os.path.join(BASE_DIR, "sroie_gold_text")
    elif ENGINE == "tesseract":
        TEXT_DIR = os.path.join(BASE_DIR, "sroie_ocr", "tesseract")
    elif ENGINE == "paddleocr":
        TEXT_DIR = os.path.join(BASE_DIR, "sroie_ocr", "paddleocr")
    elif ENGINE == "easyocr":
        TEXT_DIR = os.path.join(BASE_DIR, "sroie_ocr", "easyocr")
    else:
        raise ValueError(f"Unknown ENGINE={ENGINE}")

    SHOT_VARIANT = "0shot"  # "0shot" | "1shot" | "2shot" | "3shot"
    if SHOT_VARIANT not in {"0shot", "1shot", "2shot", "3shot"}:
        raise ValueError(f"Unsupported SHOT_VARIANT={SHOT_VARIANT}, use one of 0/1/2/3.")

    RESUME = True

    RESULTS_BASE = "/content/drive/MyDrive/data/pixels_to_pairs_final_results/sroie"
    os.makedirs(RESULTS_BASE, exist_ok=True)

    MAX_DOCS         = None   # None = all docs
    MAX_NEW_TOKENS   = 1024  # fixed across all final benchmark runs
    MAX_INPUT_TOKENS = 8192

    BATCH_SIZE       = 1
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    DEBUG_FIRST_DOC = False

    # =====================================================
    # 4. MODEL LIST – EDIT THIS TO CHOOSE MODELS
    # =====================================================

    MODEL_SPECS = [
        {"name": "Qwen/Qwen2.5-7B-Instruct", "family": "causal"},
        {"name": "meta-llama/Meta-Llama-3-8B-Instruct", "family": "causal"},
        {"name": "mistralai/Mistral-7B-Instruct-v0.2", "family": "causal"},
        {"name": "google/gemma-2b-it", "family": "causal"},
        {"name": "google/gemma-7b-it", "family": "causal"},
    ]

    # =====================================================
    # 5. TEXT UTILS
    # =====================================================

    def normalize_str(s: str) -> str:
        return " ".join(str(s).strip().lower().split())


    def normalize_key(s: str) -> str:
        """
        Normalize field labels for evaluation only.

        This is intentionally formatting-tolerant but not semantic/fuzzy:
        - Unicode NFKC normalization
        - case-insensitive matching via casefold()
        - trim surrounding whitespace
        - ignore common label punctuation/decoration
        - treat slash, backslash, hyphen, underscore, and vertical bar as word separators
        - collapse repeated whitespace

        Examples that become equivalent:
          "DATE:" == "date"
          "• CASE NAME :" == "Case Name"
          "NO. OF STORES" == "no of stores"
          "P.O.S." == "POS"
          "SENDER /PHONE NUMBER:" == "sender/phone number"

        No synonym expansion, fuzzy matching, stemming, or answer repair is used.
        """
        import unicodedata

        s = unicodedata.normalize("NFKC", str(s)).casefold().strip()
        if not s:
            return ""

        # Drop purely decorative leading/trailing characters.
        while s and not s[0].isalnum():
            s = s[1:]
        while s and not s[-1].isalnum():
            s = s[:-1]

        if not s:
            return ""

        out = []
        separators = {"/", "\\", "-", "_", "|"}
        removable_punct = {
            ".", ",", ":", ";", "'", '"', "`",
            "(", ")", "[", "]", "{", "}", "<", ">",
            "!", "?", "*", "#", "~", "^"
        }

        for ch in s:
            if ch.isalnum():
                out.append(ch)
            elif ch.isspace() or ch in separators:
                out.append(" ")
            elif ch in removable_punct:
                # Formatting punctuation inside labels is ignored.
                continue
            else:
                # Preserve non-formatting symbols (e.g., &, +, $) as tokens
                # so genuinely different labels are not silently merged.
                out.append(f" {ch} ")

        return " ".join("".join(out).split())

    def text_f1(gt: str, pred: str) -> float:
        """Token-level F1 with multiplicities (Counter overlap)."""
        gt_tokens = normalize_str(gt).split()
        pred_tokens = normalize_str(pred).split()
        if not gt_tokens and not pred_tokens:
            return 1.0
        if not gt_tokens or not pred_tokens:
            return 0.0

        gt_c = Counter(gt_tokens)
        pred_c = Counter(pred_tokens)
        overlap = sum((gt_c & pred_c).values())
        if overlap == 0:
            return 0.0
        precision = overlap / len(pred_tokens)
        recall    = overlap / len(gt_tokens)
        return 2 * precision * recall / (precision + recall)

    # =====================================================
    # 6. LOAD SROIE KVP GROUND TRUTH
    # =====================================================

    def load_sroie_gt_kvp(kvp_dir: str) -> Dict[str, List[Dict[str, str]]]:
        """
        Expects each JSON file to be either:
          1) {"kvp": [ {"key": "company", "value": "..."}, ... ]}
          2) [ {"key": "company", "value": "..."}, ... ]
        """
        gt: Dict[str, List[Dict[str, str]]] = {}

        files = sorted(f for f in os.listdir(kvp_dir) if f.endswith(".json"))
        print(f"Found {len(files)} annotation files in {kvp_dir}")

        for fname in files:
            doc_id = os.path.splitext(fname)[0]
            fpath = os.path.join(kvp_dir, fname)
            with open(fpath, "r", encoding="utf-8") as f:
                data = json.load(f)

            if isinstance(data, dict) and "kvp" in data:
                kvps = data["kvp"]
            elif isinstance(data, list):
                kvps = data
            else:
                print(f"[WARN] Unknown SROIE annotation format for {fname}, skipping.")
                continue

            # Normalize to list of {"key": str, "value": str}
            clean_list: List[Dict[str, str]] = []
            for d in kvps:
                if not isinstance(d, dict):
                    continue
                key = str(d.get("key", "")).strip()
                val = str(d.get("value", "")).strip()
                if key == "" and val == "":
                    continue
                clean_list.append({"key": key, "value": val})

            gt[doc_id] = clean_list

        print(f"Loaded KVP ground truth for {len(gt)} SROIE documents.")
        return gt

    # =====================================================
    # 7. LOAD TEXT FILES (SROIE)
    # =====================================================

    def load_doc_texts(text_dir: str) -> Dict[str, str]:
        texts: Dict[str, str] = {}
        files = sorted(f for f in os.listdir(text_dir) if f.endswith(".txt"))
        print(f"Found {len(files)} text files in {text_dir}")

        for fname in files:
            doc_id = os.path.splitext(fname)[0]
            fpath = os.path.join(text_dir, fname)
            with open(fpath, "r", encoding="utf-8") as f:
                doc_text = f.read()
            texts[doc_id] = doc_text

        return texts

    # =====================================================
    # 8. PROMPT (SROIE, RECEIPTS, FEW-SHOT)
    # =====================================================

    SYSTEM_MESSAGE = (
        "You extract key–value pairs from OCR text of receipts and invoices. "
        "Return ONLY a JSON object with the exact schema "
        "{\"kvp\": [{\"key\":..., \"value\":...}, ...]}. "
        "Do not output any extra text."
    )

    # Base instructions used for ALL shots (including 0-shot)
    USER_INSTRUCTIONS_BASE = """
    Task: From the OCR text of a single receipt or invoice, extract the following fields if present:

    - company  : the merchant/store name
    - address  : the full business address
    - date     : the transaction date
    - total    : the final total amount paid (prefer the grand total / inclusive-of-tax amount)

    Output rules (strict):
    - Output ONLY valid JSON (no markdown, no code fences, no explanations).
    - JSON must be exactly: {"kvp": [{"key": "...", "value": "..."}, ...]}
    - Keys should be lower-case strings from this set whenever possible:
        ["company", "address", "date", "total"]
    - Values should be the best string span for that field (do NOT invent values).
    - If a field is missing, simply omit it from "kvp".
    - If nothing is extractable: {"kvp": []}
    """.strip()

    # Few-shot **extension** (ONLY used for 1/2/3-shot)
    USER_INSTRUCTIONS_FEWSHOT = (USER_INSTRUCTIONS_BASE + """

    CRITICAL anti-anchoring rules:
    - The examples are ONLY to demonstrate the output format.
    - NEVER copy example values into the new output.
    - For the new receipt, extract fields ONLY from the text inside <NEW_DOCUMENT>.
    - Ignore all text inside any <EXAMPLE_*> blocks.
    """).strip()

    # ---------- SROIE few-shot examples (loaded from train-set files) ----------

    EXAMPLE_FILES_DIR = "/content/drive/MyDrive/data/sroie/sroie_prompt_examples"

    EX1_TXT = os.path.join(EXAMPLE_FILES_DIR, "X51008142065.txt")
    EX1_KVP = os.path.join(EXAMPLE_FILES_DIR, "X51008142065_kvp.json")

    EX2_TXT = os.path.join(EXAMPLE_FILES_DIR, "X51008164525.txt")
    EX2_KVP = os.path.join(EXAMPLE_FILES_DIR, "X51008164525_kvp.json")

    EX3_TXT = os.path.join(EXAMPLE_FILES_DIR, "X51008030566.txt")
    EX3_KVP = os.path.join(EXAMPLE_FILES_DIR, "X51008030566_kvp.json")

    def _load_prompt_example(
        doc_id: str,
        txt_path: str,
        kvp_path: str,
    ) -> Dict[str, Any]:
        """
        Load one frozen SROIE few-shot example from disk.

        The document text is stored in <doc_id>.txt.
        The KVP annotation is stored in <doc_id>_kvp.json and must contain
        valid JSON in one of these forms:

          1) {"kvp": [{"key": "...", "value": "..."}, ...]}
          2) [{"key": "...", "value": "..."}, ...]

        These examples must come from the SROIE training split.
        """
        if not os.path.isfile(txt_path):
            raise FileNotFoundError(
                f"Missing SROIE prompt-example text file for {doc_id}: {txt_path}"
            )

        if not os.path.isfile(kvp_path):
            raise FileNotFoundError(
                f"Missing SROIE prompt-example KVP file for {doc_id}: {kvp_path}"
            )

        with open(txt_path, "r", encoding="utf-8") as f:
            doc_text = f.read().strip()

        with open(kvp_path, "r", encoding="utf-8") as f:
            kvp_text = f.read().strip()

        if not doc_text:
            raise ValueError(
                f"Prompt-example document text is empty for {doc_id}: {txt_path}"
            )

        if not kvp_text:
            raise ValueError(
                f"Prompt-example KVP file is empty for {doc_id}: {kvp_path}"
            )

        try:
            data = json.loads(kvp_text)
        except json.JSONDecodeError as e:
            raise ValueError(
                f"Prompt-example KVP file is not valid JSON for {doc_id}: {kvp_path}"
            ) from e

        if isinstance(data, dict) and "kvp" in data:
            kvp_raw = data["kvp"]
        elif isinstance(data, list):
            kvp_raw = data
        else:
            raise ValueError(
                f"Unsupported prompt-example KVP format for {doc_id}: {kvp_path}"
            )

        if not isinstance(kvp_raw, list):
            raise ValueError(
                f"Prompt-example KVP content must be a list for {doc_id}: {kvp_path}"
            )

        kvp_clean: List[Dict[str, str]] = []

        for i, pair in enumerate(kvp_raw):
            if not isinstance(pair, dict):
                raise ValueError(
                    f"Prompt example {doc_id}, KVP item {i} is not a JSON object."
                )

            key = pair.get("key")
            value = pair.get("value")

            if not isinstance(key, str) or not isinstance(value, str):
                raise ValueError(
                    f"Prompt example {doc_id}, KVP item {i} must contain "
                    f"string-valued 'key' and 'value' fields."
                )

            key = key.strip()
            value = value.strip()

            if not key or not value:
                raise ValueError(
                    f"Prompt example {doc_id}, KVP item {i} has an empty key/value."
                )

            kvp_clean.append({
                "key": key,
                "value": value,
            })

        if not kvp_clean:
            raise ValueError(
                f"Prompt-example KVP list is empty for {doc_id}: {kvp_path}"
            )

        return {
            "doc_id": doc_id,
            "doc_text": doc_text,
            "kvp": kvp_clean,
        }

    # Frozen training examples.
    # Ordering defines the composition of 1-shot, 2-shot, and 3-shot prompts.
    SROIE_EXAMPLES = [
        _load_prompt_example("X51008142065", EX1_TXT, EX1_KVP),
        _load_prompt_example("X51008164525", EX2_TXT, EX2_KVP),
        _load_prompt_example("X51008030566", EX3_TXT, EX3_KVP),
    ]

    EXPECTED_EXAMPLE_IDS = [
        "X51008142065",
        "X51008164525",
        "X51008030566",
    ]

    loaded_example_ids = [ex["doc_id"] for ex in SROIE_EXAMPLES]

    assert loaded_example_ids == EXPECTED_EXAMPLE_IDS, (
        f"Unexpected SROIE few-shot example order: {loaded_example_ids}"
    )

    def _example_block(idx: int, doc_text: str, kvp_list: List[Dict[str, str]]) -> str:
        json_obj = {"kvp": kvp_list}
        json_str = json.dumps(json_obj, ensure_ascii=False, indent=2)
        return (
            f"<EXAMPLE_{idx}_DOCUMENT>\n{doc_text}\n</EXAMPLE_{idx}_DOCUMENT>\n"
            f"<EXAMPLE_{idx}_OUTPUT>\n{json_str}\n</EXAMPLE_{idx}_OUTPUT>\n"
        )

    def _task_footer(doc_text: str) -> str:
        return (
            "Now do the task for the NEW document below.\n"
            "Extract fields ONLY from <NEW_DOCUMENT>. Ignore all <EXAMPLE_*> blocks completely.\n"
            "Return ONLY a SINGLE JSON object and stop immediately after the final '}'.\n"
            "<NEW_DOCUMENT>\n"
            f"{doc_text}\n"
            "</NEW_DOCUMENT>\n"
            "Output:\n"
        )

    def build_prompt_0shot(doc_text: str) -> str:
        prompt = USER_INSTRUCTIONS_BASE + "\n\n" + _task_footer(doc_text)
        # Guard sanity: 0-shot must not contain example blocks
        assert "<EXAMPLE_1_DOCUMENT>" not in prompt, "0-shot prompt accidentally contains example blocks."
        return prompt

    def build_prompt_1shot(doc_text: str) -> str:
        ex1 = SROIE_EXAMPLES[0]
        return "\n\n".join([
            USER_INSTRUCTIONS_FEWSHOT,
            _example_block(1, ex1["doc_text"], ex1["kvp"]),
            _task_footer(doc_text),
        ])

    def build_prompt_2shot(doc_text: str) -> str:
        ex1 = SROIE_EXAMPLES[0]
        ex2 = SROIE_EXAMPLES[1]
        return "\n\n".join([
            USER_INSTRUCTIONS_FEWSHOT,
            _example_block(1, ex1["doc_text"], ex1["kvp"]),
            _example_block(2, ex2["doc_text"], ex2["kvp"]),
            _task_footer(doc_text),
        ])

    def build_prompt_3shot(doc_text: str) -> str:
        ex1 = SROIE_EXAMPLES[0]
        ex2 = SROIE_EXAMPLES[1]
        ex3 = SROIE_EXAMPLES[2]
        return "\n\n".join([
            USER_INSTRUCTIONS_FEWSHOT,
            _example_block(1, ex1["doc_text"], ex1["kvp"]),
            _example_block(2, ex2["doc_text"], ex2["kvp"]),
            _example_block(3, ex3["doc_text"], ex3["kvp"]),
            _task_footer(doc_text),
        ])

    def build_prompt(doc_text: str) -> str:
        if SHOT_VARIANT == "0shot":
            prompt = build_prompt_0shot(doc_text)
            expected_shots = 0
        elif SHOT_VARIANT == "1shot":
            prompt = build_prompt_1shot(doc_text)
            expected_shots = 1
        elif SHOT_VARIANT == "2shot":
            prompt = build_prompt_2shot(doc_text)
            expected_shots = 2
        elif SHOT_VARIANT == "3shot":
            prompt = build_prompt_3shot(doc_text)
            expected_shots = 3
        else:
            raise ValueError(f"Unknown SHOT_VARIANT: {SHOT_VARIANT}")

        # Sanity guard: ensure the prompt contains exactly the intended
        # cumulative number of example-document blocks.
        for idx in range(1, 4):
            tag = f"<EXAMPLE_{idx}_DOCUMENT>"
            if idx <= expected_shots:
                assert tag in prompt, (
                    f"{SHOT_VARIANT} prompt is missing expected example block {idx}."
                )
            else:
                assert tag not in prompt, (
                    f"{SHOT_VARIANT} prompt unexpectedly contains example block {idx}."
                )

        return prompt

    # =====================================================
    # 9. PARSING
    # =====================================================

    def parse_kvp_output(raw_output: str) -> List[Dict[str, str]]:
        """
        Parse LLM output conservatively while preserving complete KVP objects.

        1. Prefer a complete valid JSON response:
           {"kvp": [{"key": "...", "value": "..."}, ...]}
        2. If the full response is malformed/truncated, recover only individually
           COMPLETE, valid JSON objects already present in the generated text
           that contain string-valued "key" and "value" fields.

        No regex KVP extraction, malformed-JSON repair, guessing, fuzzy parsing,
        GT-aware recovery, or document-text fallback is performed.
        """
        if not isinstance(raw_output, str):
            return []

        s = raw_output.strip()
        if not s:
            return []

        if s.startswith("```"):
            lines = s.splitlines()
            if lines and lines[0].strip().startswith("```"):
                lines = lines[1:]
            if lines and lines[-1].strip() == "```":
                lines = lines[:-1]
            s = "\n".join(lines).strip()

        try:
            obj = json.loads(s)
        except json.JSONDecodeError:
            obj = None

        if isinstance(obj, dict):
            kvp_list = obj.get("kvp")
            if isinstance(kvp_list, list):
                parsed = []
                for pair in kvp_list:
                    if not isinstance(pair, dict):
                        continue
                    key = pair.get("key")
                    value = pair.get("value")
                    if not isinstance(key, str) or not isinstance(value, str):
                        continue
                    key = key.strip()
                    value = value.strip()
                    if key == "" and value == "":
                        continue
                    parsed.append({"key": key, "value": value})
                return parsed

        decoder = json.JSONDecoder()
        recovered = []

        for start, char in enumerate(s):
            if char != "{":
                continue
            try:
                candidate, _ = decoder.raw_decode(s[start:])
            except json.JSONDecodeError:
                continue

            if not isinstance(candidate, dict):
                continue

            key = candidate.get("key")
            value = candidate.get("value")
            if not isinstance(key, str) or not isinstance(value, str):
                continue

            key = key.strip()
            value = value.strip()
            if key == "" and value == "":
                continue

            recovered.append({"key": key, "value": value})

        return recovered

    # =====================================================
    # 10. METRICS (same as FUNSD)
    # =====================================================

    # NOTE ON METRICS:
    # - Key Recall is duplicate-aware and uses formatting-tolerant normalized keys.
    # - Exact Match requires the normalized key and normalized value to match exactly.
    # - Value F1 is token-overlap F1 on values, matched by normalized key with no prediction reuse.
    #   Every GT pair contributes to the final average; missing keys and zero-overlap values receive 0.
    #   This implementation is intentionally aligned with the manuscript definition.

    def compute_doc_metrics(
        gt_kvp: List[Dict[str, str]],
        pred_kvp: List[Dict[str, str]]
    ) -> Dict[str, Any]:

        num_gt = len(gt_kvp)

        if num_gt == 0:
            return {
                "num_gt_keys": 0,
                "matched_keys": 0,
                "key_recall": 0.0,
                "exact_matches": 0,
                "exact_match_rate": 0.0,
                "value_f1": 0.0,
            }

        # -------------------------------------------------
        # Normalize GT and predicted key/value strings
        # -------------------------------------------------
        gt_keys_norm = [
            normalize_key(kv.get("key", ""))
            for kv in gt_kvp
        ]
        gt_vals_norm = [
            normalize_str(kv.get("value", ""))
            for kv in gt_kvp
        ]

        pred_keys_norm = [
            normalize_key(kv.get("key", ""))
            for kv in pred_kvp
        ]
        pred_vals_norm = [
            normalize_str(kv.get("value", ""))
            for kv in pred_kvp
        ]

        # =================================================
        # 1. KEY RECALL
        # Duplicate-aware key matching
        # =================================================

        gt_key_counts = Counter(k for k in gt_keys_norm if k)
        pred_key_counts = Counter(k for k in pred_keys_norm if k)

        matched_keys = sum(
            min(gt_key_counts[k], pred_key_counts.get(k, 0))
            for k in gt_key_counts
        )

        key_recall = matched_keys / num_gt

        # =================================================
        # 2. EXACT MATCH RATE
        # Exact normalized key AND exact normalized value.
        # Each prediction can be used at most once.
        # =================================================

        used_pred_exact = set()
        exact_matches = 0

        for gt_key, gt_value in zip(gt_keys_norm, gt_vals_norm):

            if not gt_key or not gt_value:
                continue

            for j, (pred_key, pred_value) in enumerate(
                zip(pred_keys_norm, pred_vals_norm)
            ):
                if j in used_pred_exact:
                    continue

                if gt_key == pred_key and gt_value == pred_value:
                    exact_matches += 1
                    used_pred_exact.add(j)
                    break

        exact_match_rate = exact_matches / num_gt

        # =================================================
        # 3. VALUE TOKEN F1
        #
        # For every GT pair:
        #   - consider unused predictions with the same key
        #   - select the one with maximum token-level F1
        #   - same-key but zero-overlap value -> F1 = 0
        #   - no same-key prediction -> F1 = 0
        #
        # Final Value F1 is averaged over ALL GT pairs.
        # =================================================

        used_pred_f1 = set()
        value_f1_scores = []

        for gt_pair in gt_kvp:

            gt_key = normalize_key(gt_pair.get("key", ""))
            gt_value = str(gt_pair.get("value", ""))

            # A malformed/empty GT field receives zero credit.
            if not gt_key or not normalize_str(gt_value):
                value_f1_scores.append(0.0)
                continue

            best_j = None
            best_f1 = 0.0

            for j, pred_pair in enumerate(pred_kvp):

                if j in used_pred_f1:
                    continue

                pred_key = normalize_key(pred_pair.get("key", ""))

                if pred_key != gt_key or not pred_key:
                    continue

                pred_value = str(pred_pair.get("value", ""))

                f1 = text_f1(gt_value, pred_value)

                # Important:
                # best_j must also be assigned when F1 == 0.
                if best_j is None or f1 > best_f1:
                    best_j = j
                    best_f1 = f1

            if best_j is not None:
                used_pred_f1.add(best_j)
                value_f1_scores.append(best_f1)
            else:
                # Missing key
                value_f1_scores.append(0.0)

        value_f1 = float(np.mean(value_f1_scores))

        return {
            "num_gt_keys": num_gt,
            "matched_keys": int(matched_keys),
            "key_recall": float(key_recall),
            "exact_matches": int(exact_matches),
            "exact_match_rate": float(exact_match_rate),
            "value_f1": float(value_f1),
        }

    # =====================================================
    # 11. MODEL LOADING + GENERATION (NO BNB)
    # =====================================================

    def _preferred_dtype() -> torch.dtype:
        if DEVICE == "cuda":
            return torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
        return torch.float32

    def build_model_and_tokenizer(model_name: str, family: str):
        print(f"Loading model: {model_name}  (family={family})")

        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()

        kwargs = {}
        if HF_TOKEN:
            kwargs["token"] = HF_TOKEN

        config = AutoConfig.from_pretrained(model_name, **kwargs)
        tokenizer = AutoTokenizer.from_pretrained(model_name, **kwargs)

        is_encoder_decoder = bool(getattr(config, "is_encoder_decoder", False)) or (family == "seq2seq")

        if not is_encoder_decoder:
            tokenizer.padding_side = "left"

        dtype = _preferred_dtype()

        if DEVICE == "cuda":
            primary_device_map = "cuda"
        else:
            primary_device_map = None

        def _load(device_map):
            if is_encoder_decoder:
                return AutoModelForSeq2SeqLM.from_pretrained(
                    model_name,
                    device_map=device_map,
                    dtype=dtype,
                    **kwargs,
                )
            return AutoModelForCausalLM.from_pretrained(
                model_name,
                device_map=device_map,
                dtype=dtype,
                **kwargs,
            )

        try:
            model = _load(primary_device_map)
        except Exception as e:
            # Do not silently change device placement. A failed CUDA load must be
            # surfaced so the run cannot continue under a different hardware
            # configuration (for example CPU offloading via device_map="auto").
            raise RuntimeError(
                f"Failed to load {model_name} with device_map={primary_device_map!r}. "
                "Check local disk space/model cache and GPU memory before retrying."
            ) from e

        if tokenizer.pad_token is None and tokenizer.eos_token is not None:
            tokenizer.pad_token = tokenizer.eos_token
            model.config.pad_token_id = tokenizer.eos_token_id

        print("hf_device_map:", getattr(model, "hf_device_map", None))
        model.eval()
        return tokenizer, model, is_encoder_decoder

    def _effective_max_input_tokens(tokenizer) -> int:
        mml = getattr(tokenizer, "model_max_length", None)
        if mml is None or mml > 100000:
            return MAX_INPUT_TOKENS
        return int(min(MAX_INPUT_TOKENS, mml))

    def _maybe_apply_chat_template(tokenizer, user_prompt: str) -> str:
        try:
            tmpl = getattr(tokenizer, "chat_template", None)
            if tmpl:
                messages = [
                    {"role": "system", "content": SYSTEM_MESSAGE},
                    {"role": "user", "content": user_prompt},
                ]
                return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        except Exception:
            pass
        return user_prompt

    def _qwen_im_end_eos_id(tokenizer) -> Optional[int]:
        try:
            im_end_id = tokenizer.convert_tokens_to_ids("<|im_end|>")
            if im_end_id is not None and im_end_id != tokenizer.unk_token_id:
                return int(im_end_id)
        except Exception:
            pass
        return None

    def _postprocess_decoded_text(s: str) -> str:
        s = s.strip()
        s = s.replace("<|im_end|>", "").strip()
        return s

    def run_model_on_prompts(tokenizer, model, prompts: List[str], is_encoder_decoder: bool):
        """Generate outputs plus neutral generation-length diagnostics."""
        if not is_encoder_decoder:
            prompts = [_maybe_apply_chat_template(tokenizer, p) for p in prompts]

        max_in = _effective_max_input_tokens(tokenizer)

        inputs = tokenizer(
            prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=max_in,
        )

        if DEVICE == "cuda":
            inputs = {k: v.to("cuda") for k, v in inputs.items()}

        eos_id = _qwen_im_end_eos_id(tokenizer) or int(tokenizer.eos_token_id)

        gen_kwargs = dict(
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=int(tokenizer.pad_token_id),
            eos_token_id=int(eos_id),
            return_dict_in_generate=True,
            output_scores=False,
        )

        with torch.no_grad():
            gen_out = model.generate(**inputs, **gen_kwargs)

        sequences = gen_out.sequences
        results = []

        if is_encoder_decoder:
            for i in range(sequences.shape[0]):
                ids = sequences[i]
                generated_tokens = int(ids.shape[-1])
                text = tokenizer.decode(ids, skip_special_tokens=True).strip()
                results.append({
                    "text": text,
                    "generated_tokens": generated_tokens,
                    "output_hit_max_new_tokens": bool(generated_tokens >= MAX_NEW_TOKENS),
                })
            return results

        input_width = int(inputs["input_ids"].shape[1])

        for i in range(sequences.shape[0]):
            ids = sequences[i, input_width:]
            generated_tokens = int(ids.shape[-1])
            text = tokenizer.decode(ids, skip_special_tokens=False)
            text = _postprocess_decoded_text(text)

            results.append({
                "text": text,
                "generated_tokens": generated_tokens,
                "output_hit_max_new_tokens": bool(generated_tokens >= MAX_NEW_TOKENS),
            })

        return results

    # =====================================================
    # 12. PROMPT TOKEN LOGGING
    # =====================================================

    def prompt_token_stats(
        tokenizer,
        prompt: str,
        is_encoder_decoder: bool,
    ) -> Tuple[int, int, bool]:
        """
        Measure token length on the same prompt representation used for generation.

        For causal/chat models, apply the tokenizer chat template before measuring.
        For encoder-decoder models, measure the raw task prompt directly.

        This affects diagnostics only; it does not alter generation.
        """
        max_in = _effective_max_input_tokens(tokenizer)

        prompt_for_model = (
            prompt
            if is_encoder_decoder
            else _maybe_apply_chat_template(tokenizer, prompt)
        )

        tok_no_trunc = tokenizer(
            prompt_for_model,
            return_tensors="pt",
            truncation=False,
        )

        tok_trunc = tokenizer(
            prompt_for_model,
            return_tensors="pt",
            truncation=True,
            max_length=max_in,
        )

        n_no_trunc = int(tok_no_trunc["input_ids"].shape[-1])
        n_trunc = int(tok_trunc["input_ids"].shape[-1])

        return n_no_trunc, n_trunc, (n_no_trunc > n_trunc)

    # =====================================================
    # 13. MAIN EVAL LOOP (SROIE)
    # =====================================================

    def _cleanup_cuda():
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()

    def evaluate_model_on_sroie(model_name: str, family: str):
        tokenizer = None
        model = None
        is_encoder_decoder = False

        try:
            print("\n" + "=" * 60)
            print(f"=== Running model {model_name} on SROIE ({ENGINE}, {SHOT_VARIANT}) ===")

            gt_dict   = load_sroie_gt_kvp(KVP_DIR)
            text_dict = load_doc_texts(TEXT_DIR)

            doc_ids = sorted(set(gt_dict.keys()) & set(text_dict.keys()))
            if MAX_DOCS is not None:
                doc_ids = doc_ids[:MAX_DOCS]
            print(f"Total docs available: {len(doc_ids)}")

            out_dir_name = f"sroie_{ENGINE}_{SHOT_VARIANT}_{model_name.replace('/', '_')}"
            OUTPUT_DIR   = os.path.join(RESULTS_BASE, out_dir_name)
            os.makedirs(OUTPUT_DIR, exist_ok=True)
            print(f"Results will be saved under: {OUTPUT_DIR}")

            csv_path  = os.path.join(OUTPUT_DIR, f"metrics_{ENGINE}_{SHOT_VARIANT}_{model_name.replace('/', '_')}.csv")
            pred_path = os.path.join(OUTPUT_DIR, f"predictions_{ENGINE}_{SHOT_VARIANT}_{model_name.replace('/', '_')}.jsonl")

            if not RESUME:
                if os.path.exists(csv_path):
                    os.remove(csv_path)
                if os.path.exists(pred_path):
                    os.remove(pred_path)

            processed_ids = set()
            csv_exists = os.path.exists(csv_path)
            if RESUME and csv_exists:
                with open(csv_path, "r", encoding="utf-8") as f:
                    lines = f.readlines()
                if len(lines) > 1:
                    for line in lines[1:]:
                        parts = line.strip().split(",")
                        if parts and parts[0]:
                            processed_ids.add(parts[0])
                print(f"Found existing metrics CSV with {len(processed_ids)} docs. Resuming from there.")

            doc_ids_to_run = [d for d in doc_ids if d not in processed_ids]
            print(f"Docs remaining for this model: {len(doc_ids_to_run)}")

            if len(doc_ids_to_run) != 0:
                try:
                    tokenizer, model, is_encoder_decoder = build_model_and_tokenizer(model_name, family)
                except Exception as e:
                    print(f"[ERROR] Could not load model {model_name}: {e}")
                    print("Skipping this model.\n")
                    return

                csv_mode = "a" if (RESUME and csv_exists) else "w"
                with open(csv_path, csv_mode, encoding="utf-8") as csv_f, \
                     open(pred_path, "a", encoding="utf-8") as pred_f:

                    if csv_mode == "w":
                        csv_f.write("doc_id,num_gt_keys,matched_keys,key_recall,exact_match_rate,value_f1,prompt_tokens_no_trunc,prompt_tokens_trunc,prompt_truncated,generated_tokens,output_hit_max_new_tokens\n")

                    with tqdm(
                        total=len(doc_ids_to_run),
                        desc=f"{model_name} [{ENGINE}][{SHOT_VARIANT}]",
                        dynamic_ncols=True
                    ) as pbar:

                        for start in range(0, len(doc_ids_to_run), BATCH_SIZE):
                            batch_ids    = doc_ids_to_run[start:start + BATCH_SIZE]
                            batch_texts  = [text_dict[d] for d in batch_ids]
                            batch_gt_kvp = [gt_dict[d] for d in batch_ids]

                            batch_prompts = [build_prompt(t) for t in batch_texts]
                            batch_prompt_stats = [
                                prompt_token_stats(tokenizer, p, is_encoder_decoder)
                                for p in batch_prompts
                            ]

                            if DEBUG_FIRST_DOC and start == 0:
                                p0 = batch_prompts[0]
                                max_in = _effective_max_input_tokens(tokenizer)
                                n_no_trunc, n_trunc, truncated = batch_prompt_stats[0]
                                print("\n===== DEBUG: PROMPT TOKEN ANALYSIS =====")
                                print(f"Tokenizer model_max_length: {getattr(tokenizer, 'model_max_length', None)}")
                                print(f"Effective max_input_tokens: {max_in}")
                                print(f"Prompt tokens (no trunc):   {n_no_trunc}")
                                print(f"Prompt tokens (truncated):   {n_trunc}")
                                if truncated:
                                    print("⚠️ Prompt was truncated.")
                                print("Contains <NEW_DOCUMENT> footer:", "<NEW_DOCUMENT>" in p0 and "</NEW_DOCUMENT>" in p0)
                                print("\n----- PROMPT TAIL (last 1200 chars) -----")
                                print(p0[-1200:])
                                print("===== END DEBUG =====\n")

                            generation_results = run_model_on_prompts(tokenizer, model, batch_prompts, is_encoder_decoder)

                            for (doc_id, doc_text, gt_kvp, gen_result, pstats) in zip(
                                batch_ids, batch_texts, batch_gt_kvp, generation_results, batch_prompt_stats
                            ):
                                n_no_trunc, n_trunc, truncated = pstats
                                raw_output = gen_result["text"]
                                generated_tokens = int(gen_result["generated_tokens"])
                                output_hit_max_new_tokens = bool(gen_result["output_hit_max_new_tokens"])

                                pred_kvp = parse_kvp_output(raw_output)
                                metrics  = compute_doc_metrics(gt_kvp, pred_kvp)

                                csv_f.write(
                                    f"{doc_id},{metrics['num_gt_keys']},{metrics['matched_keys']},"
                                    f"{metrics['key_recall']:.10f},{metrics['exact_match_rate']:.10f},"
                                    f"{metrics['value_f1']:.10f},{n_no_trunc},{n_trunc},{int(truncated)},"
                                    f"{generated_tokens},{int(output_hit_max_new_tokens)}\n"
                                )
                                csv_f.flush()
                                os.fsync(csv_f.fileno())

                                record = {
                                    "doc_id": doc_id,
                                    "engine": ENGINE,
                                    "shot_variant": SHOT_VARIANT,
                                    "model": model_name,
                                    "gt_kvp": gt_kvp,
                                    "pred_kvp": pred_kvp,
                                    "raw_output": raw_output,
                                    "metrics": metrics,
                                    "prompt_tokens_no_trunc": n_no_trunc,
                                    "prompt_tokens_trunc": n_trunc,
                                    "prompt_truncated": bool(truncated),
                                    "generated_tokens": generated_tokens,
                                    "output_hit_max_new_tokens": output_hit_max_new_tokens,
                                    "seed": SEED,
                                    "dtype": str(_preferred_dtype()),
                                    "fewshot_example_ids": [
                                        ex["doc_id"]
                                        for ex in SROIE_EXAMPLES[
                                            :(
                                                int(SHOT_VARIANT[0])
                                                if SHOT_VARIANT != "0shot"
                                                else 0
                                            )
                                        ]
                                    ],
                                }
                                pred_f.write(json.dumps(record) + "\n")
                                pred_f.flush()
                                os.fsync(pred_f.fileno())

                                pbar.update(1)

            # ---------- Final summary from CSV ----------
            macro_key_recall = 0.0
            macro_em         = 0.0
            macro_value_f1   = 0.0
            n_docs           = 0

            if os.path.exists(csv_path):
                with open(csv_path, "r", encoding="utf-8") as f:
                    lines = f.readlines()
                if len(lines) > 1:
                    sum_key_rec = 0.0
                    sum_em      = 0.0
                    sum_f1      = 0.0
                    for line in lines[1:]:
                        parts = line.strip().split(",")
                        if len(parts) < 6:
                            continue
                        try:
                            key_rec = float(parts[3])
                            em      = float(parts[4])
                            f1      = float(parts[5])
                        except ValueError:
                            continue
                        sum_key_rec += key_rec
                        sum_em      += em
                        sum_f1      += f1
                        n_docs      += 1
                    if n_docs > 0:
                        macro_key_recall = sum_key_rec / n_docs
                        macro_em         = sum_em / n_docs
                        macro_value_f1   = sum_f1 / n_docs

            print("\n==== SUMMARY (MACRO AVERAGE OVER DOCS) ====")
            print("Dataset: SROIE")
            print(f"Engine:  {ENGINE}")
            print(f"Shot:    {SHOT_VARIANT}")
            print(f"Model:   {model_name}")
            print(f"Docs (from CSV): {n_docs}")
            print(f"Key Recall: {macro_key_recall:.4f}")
            print(f"EM:         {macro_em:.4f}")
            print(f"Value F1:   {macro_value_f1:.4f}")
            print(f"\nPer-doc metrics saved to: {csv_path}")
            print(f"Predictions saved to:      {pred_path}")
            print("=" * 60 + "\n")

        finally:
            try:
                if model is not None:
                    del model
                if tokenizer is not None:
                    del tokenizer
            except Exception:
                pass
            _cleanup_cuda()

    # =====================================================
    # 14. MAIN
    # =====================================================

    def main():
        for spec in tqdm(MODEL_SPECS, desc=f"Models [{SHOT_VARIANT}]", dynamic_ncols=True):
            evaluate_model_on_sroie(
                model_name=spec["name"],
                family=spec["family"],
            )

    # The runner is intentionally not auto-executed here.
    # Use the experiment cells later in the notebook.

    def configure(engine, shot, model_specs):
        """Configure one SROIE experiment without modifying any other dataset."""
        nonlocal ENGINE, SHOT_VARIANT, TEXT_DIR, MODEL_SPECS, RESUME, RESULTS_BASE

        valid_engines = {"gold_text", "tesseract", "paddleocr", "easyocr"}
        valid_shots = {"0shot", "1shot", "2shot", "3shot"}

        if engine not in valid_engines:
            raise ValueError(f"Unsupported engine: {engine}")
        if shot not in valid_shots:
            raise ValueError(f"Unsupported shot setting: {shot}")

        ENGINE = engine
        SHOT_VARIANT = shot
        MODEL_SPECS = list(model_specs)
        RESUME = True
        RESULTS_BASE = "/content/drive/MyDrive/data/pixels_to_pairs_final_results/sroie"
        os.makedirs(RESULTS_BASE, exist_ok=True)

        text_paths = {
            "gold_text": os.path.join(BASE_DIR, "sroie_gold_text"),
            "tesseract": os.path.join(BASE_DIR, "sroie_ocr", "tesseract"),
            "paddleocr": os.path.join(BASE_DIR, "sroie_ocr", "paddleocr"),
            "easyocr": os.path.join(BASE_DIR, "sroie_ocr", "easyocr"),
        }
        TEXT_DIR = text_paths[engine]

        if "/pixels_to_pairs_final_results/sroie" not in RESULTS_BASE:
            raise RuntimeError(f"SROIE results path is incorrect: {RESULTS_BASE}")
        if "/data/sroie/" not in TEXT_DIR:
            raise RuntimeError(f"SROIE text path is incorrect: {TEXT_DIR}")

        print("\nCONFIG CHECK — SROIE")
        print("  ENGINE:", ENGINE)
        print("  SHOT:", SHOT_VARIANT)
        n_shots = int(SHOT_VARIANT[0]) if SHOT_VARIANT != "0shot" else 0
        selected_example_ids = [ex["doc_id"] for ex in SROIE_EXAMPLES[:n_shots]]
        print("  FEW-SHOT EXAMPLES:", selected_example_ids)
        print("  TEXT_DIR:", TEXT_DIR)
        print("  RESULTS_BASE:", RESULTS_BASE)
        print("  MAX_NEW_TOKENS:", MAX_NEW_TOKENS)
        print("  MODELS:", [m["name"] for m in MODEL_SPECS])

    def run(engine, shot, model_specs):
        """Run one complete SROIE condition sequentially."""
        configure(engine, shot, model_specs)
        for spec in MODEL_SPECS:
            evaluate_model_on_sroie(spec["name"], spec["family"])

    def status():
        return {
            "dataset": "SROIE",
            "engine": ENGINE,
            "shot": SHOT_VARIANT,
            "text_dir": TEXT_DIR,
            "results_base": RESULTS_BASE,
            "max_new_tokens": MAX_NEW_TOKENS,
            "resume": RESUME,
        }

    return SimpleNamespace(
        configure=configure,
        run=run,
        status=status,
        evaluate=evaluate_model_on_sroie,
    )

SROIE_RUNNER = build_sroie_runner()
print("SROIE runner ready:", SROIE_RUNNER.status())


### CORD runner definition

In [ ]:
from types import SimpleNamespace

def build_cord_runner():
    # =====================================================
    # Pixels to Pairs benchmark — CORD
    # Final benchmark runner
    # =====================================================
    # This cell preserves the recovered experimental prompt/parsing/generation
    # logic for CORD, while cleaning secrets/paths and using the corrected,
    # manuscript-aligned metric implementation.
    #
    # Before running:
    #   1) Mount Google Drive using the setup cell above.
    #   2) Set HF_TOKEN in the Colab environment if a gated model requires it.
    #   3) Check ENGINE, SHOT_VARIANT, MODEL_SPECS, and dataset paths.
    #   4) Use the experiment cells later in this notebook; they run configurations sequentially.
    #
    import os
    import json
    import re
    import gc
    from typing import Dict, List, Any, Optional, Tuple
    from collections import Counter

    import numpy as np
    import torch
    from transformers import (
        AutoTokenizer,
        AutoModelForSeq2SeqLM,
        AutoModelForCausalLM,
        AutoConfig,
    )
    from tqdm.auto import tqdm

    # =====================================================
    # 0. ENV (CUDA ALLOC)
    # =====================================================

    os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

    # =====================================================
    # 1. REPRODUCIBILITY (LIGHT)
    # =====================================================

    SEED = 0
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)

    # =====================================================
    # 2. HF TOKEN & PERSISTENT CACHE
    # =====================================================

    HF_TOKEN = os.environ.get("HF_TOKEN")
    HF_CACHE_DIR = "/root/.cache/huggingface"
    os.makedirs(HF_CACHE_DIR, exist_ok=True)

    os.environ["HF_HOME"] = HF_CACHE_DIR
    os.environ["TRANSFORMERS_CACHE"] = HF_CACHE_DIR
    print(f"Using HF cache dir: {HF_CACHE_DIR}")

    # =====================================================
    # 3. CONFIG – PATHS & GLOBALS (CORD)
    # =====================================================

    def _pick_base_dir() -> str:
        """
        Auto-detect whether the mounted drive path uses gdrive or gldrive.
        """
        candidates = [
            "/content/drive/MyDrive/data/cord",
            "/content/drive/MyDrive/data/cord",
        ]
        for c in candidates:
            if os.path.isdir(c):
                return c
        return candidates[0]

    BASE_DIR = _pick_base_dir()

    KVP_DIR = os.path.join(BASE_DIR, "kvp")

    ENGINE = "gold_text"  # "gold_text" | "tesseract" | "paddleocr" | "easyocr"

    if ENGINE == "gold_text":
        TEXT_DIR = os.path.join(BASE_DIR, "cord_gold_text")
    elif ENGINE == "tesseract":
        TEXT_DIR = os.path.join(BASE_DIR, "cord_ocr", "tesseract")
    elif ENGINE == "paddleocr":
        TEXT_DIR = os.path.join(BASE_DIR, "cord_ocr", "paddleocr")
    elif ENGINE == "easyocr":
        TEXT_DIR = os.path.join(BASE_DIR, "cord_ocr", "easyocr")
    else:
        raise ValueError(f"Unknown ENGINE={ENGINE}")

    SHOT_VARIANT = "0shot"  # "0shot" | "1shot" | "2shot" | "3shot"
    if SHOT_VARIANT not in {"0shot", "1shot", "2shot", "3shot"}:
        raise ValueError(f"Unsupported SHOT_VARIANT={SHOT_VARIANT}, use one of 0/1/2/3.")

    RESUME = True

    RESULTS_BASE = "/content/drive/MyDrive/data/pixels_to_pairs_final_results/cord"
    os.makedirs(RESULTS_BASE, exist_ok=True)

    MAX_DOCS         = 100  # manuscript CORD test subset
    MAX_NEW_TOKENS   = 1024  # fixed across all final benchmark runs
    MAX_INPUT_TOKENS = 8192

    BATCH_SIZE       = 1
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    DEBUG_FIRST_DOC = False


    # =====================================================
    # 4. MODEL LIST – EDIT THIS TO CHOOSE MODELS
    # =====================================================

    MODEL_SPECS = [
        {"name": "Qwen/Qwen2.5-7B-Instruct", "family": "causal"},
        {"name": "meta-llama/Meta-Llama-3-8B-Instruct", "family": "causal"},
        {"name": "mistralai/Mistral-7B-Instruct-v0.2", "family": "causal"},
        {"name": "google/gemma-2b-it", "family": "causal"},
        {"name": "google/gemma-7b-it", "family": "causal"},
    ]

    # =====================================================
    # 5. TEXT UTILS
    # =====================================================

    def normalize_str(s: str) -> str:
        return " ".join(str(s).strip().lower().split())


    def normalize_key(s: str) -> str:
        """
        Normalize field labels for evaluation only.

        This is intentionally formatting-tolerant but not semantic/fuzzy:
        - Unicode NFKC normalization
        - case-insensitive matching via casefold()
        - trim surrounding whitespace
        - ignore common label punctuation/decoration
        - treat slash, backslash, hyphen, underscore, and vertical bar as word separators
        - collapse repeated whitespace

        Examples that become equivalent:
          "DATE:" == "date"
          "• CASE NAME :" == "Case Name"
          "NO. OF STORES" == "no of stores"
          "P.O.S." == "POS"
          "SENDER /PHONE NUMBER:" == "sender/phone number"

        No synonym expansion, fuzzy matching, stemming, or answer repair is used.
        """
        import unicodedata

        s = unicodedata.normalize("NFKC", str(s)).casefold().strip()
        if not s:
            return ""

        # Drop purely decorative leading/trailing characters.
        while s and not s[0].isalnum():
            s = s[1:]
        while s and not s[-1].isalnum():
            s = s[:-1]

        if not s:
            return ""

        out = []
        separators = {"/", "\\", "-", "_", "|"}
        removable_punct = {
            ".", ",", ":", ";", "'", '"', "`",
            "(", ")", "[", "]", "{", "}", "<", ">",
            "!", "?", "*", "#", "~", "^"
        }

        for ch in s:
            if ch.isalnum():
                out.append(ch)
            elif ch.isspace() or ch in separators:
                out.append(" ")
            elif ch in removable_punct:
                # Formatting punctuation inside labels is ignored.
                continue
            else:
                # Preserve non-formatting symbols (e.g., &, +, $) as tokens
                # so genuinely different labels are not silently merged.
                out.append(f" {ch} ")

        return " ".join("".join(out).split())

    def text_f1(gt: str, pred: str) -> float:
        gt_tokens = normalize_str(gt).split()
        pred_tokens = normalize_str(pred).split()
        if not gt_tokens and not pred_tokens:
            return 1.0
        if not gt_tokens or not pred_tokens:
            return 0.0

        gt_c = Counter(gt_tokens)
        pred_c = Counter(pred_tokens)
        overlap = sum((gt_c & pred_c).values())
        if overlap == 0:
            return 0.0
        precision = overlap / len(pred_tokens)
        recall    = overlap / len(gt_tokens)
        return 2 * precision * recall / (precision + recall)

    # =====================================================
    # 5b. DRIFT DIAGNOSTICS
    # =====================================================

    _num_like_re = re.compile(r"^[\s\W]*[\d]+([.,][\d]+)*[\s\W]*$")
    _qty_like_re = re.compile(r"^(x\s*\d+|\d+\s*x)$", re.IGNORECASE)

    def _is_numeric_like_key(k: str) -> bool:
        k = normalize_str(k)
        if not k:
            return False
        return bool(_num_like_re.match(k))

    def _is_qty_like_key(k: str) -> bool:
        k = normalize_str(k).replace(" ", "")
        if not k:
            return False
        if k.startswith("x") and k[1:].isdigit():
            return True
        if k.endswith("x") and k[:-1].isdigit():
            return True
        return False

    def compute_pred_drift_stats(pred_kvp: List[Dict[str, str]]) -> Dict[str, Any]:
        keys = [str(p.get("key", "")) for p in (pred_kvp or [])]
        keys_norm = [normalize_str(k) for k in keys if normalize_str(k)]

        if not keys_norm:
            return {
                "pred_num_pairs": 0,
                "pred_unique_keys": 0,
                "pred_dup_key_frac": 0.0,
                "pred_single_token_key_frac": 0.0,
                "pred_numeric_key_frac": 0.0,
                "pred_qty_key_frac": 0.0,
            }

        c = Counter(keys_norm)
        total = len(keys_norm)
        uniq = len(c)

        dup_count = sum(v for v in c.values() if v >= 2)
        single_token = sum(1 for k in keys_norm if len(k.split()) == 1)
        numeric_like = sum(1 for k in keys_norm if _is_numeric_like_key(k))
        qty_like = sum(1 for k in keys_norm if _is_qty_like_key(k))

        return {
            "pred_num_pairs": int(total),
            "pred_unique_keys": int(uniq),
            "pred_dup_key_frac": float(dup_count / total),
            "pred_single_token_key_frac": float(single_token / total),
            "pred_numeric_key_frac": float(numeric_like / total),
            "pred_qty_key_frac": float(qty_like / total),
        }

    # =====================================================
    # 6. SANITY HELPERS
    # =====================================================

    def _count_files(d: str, ext: str) -> int:
        if not os.path.isdir(d):
            return 0
        return sum(1 for f in os.listdir(d) if f.endswith(ext))

    def _read_head(path: str, n: int = 200) -> str:
        with open(path, "r", encoding="utf-8") as f:
            return f.read(n)

    def _global_sanity():
        print("\n================== GLOBAL SANITY ==================")
        print("BASE_DIR:", BASE_DIR)
        print("KVP_DIR: ", KVP_DIR)
        print("TEXT_DIR:", TEXT_DIR)
        print("ENGINE:  ", ENGINE)
        print("SHOT:    ", SHOT_VARIANT)
        print("MAX_DOCS:", MAX_DOCS)
        print("===================================================\n")

        if not os.path.isdir(BASE_DIR):
            raise RuntimeError(f"BASE_DIR does not exist: {BASE_DIR}")

        if not os.path.isdir(KVP_DIR):
            raise RuntimeError(
                f"KVP_DIR does not exist: {KVP_DIR}\n"
                f"Expected CORD GT in {os.path.join(BASE_DIR,'kvp')}."
            )
        if not os.path.isdir(TEXT_DIR):
            raise RuntimeError(f"TEXT_DIR does not exist: {TEXT_DIR}")

        kvp_n = _count_files(KVP_DIR, ".json")
        txt_n = _count_files(TEXT_DIR, ".txt")
        print(f"[SANITY] KVP_DIR json files: {kvp_n}")
        print(f"[SANITY] TEXT_DIR txt files: {txt_n}")

        if kvp_n == 0:
            raise RuntimeError(f"No .json files found in KVP_DIR: {KVP_DIR}")
        if txt_n == 0:
            raise RuntimeError(f"No .txt files found in TEXT_DIR: {TEXT_DIR}")

    _global_sanity()

    # =====================================================
    # 7. LOAD CORD KVP GROUND TRUTH
    # =====================================================

    def _coerce_kvp_list(data: Any) -> Optional[List[Dict[str, str]]]:
        if isinstance(data, dict) and "kvp" in data and isinstance(data["kvp"], list):
            kvps = data["kvp"]
        elif isinstance(data, list):
            kvps = data
        else:
            return None

        clean: List[Dict[str, str]] = []
        for d in kvps:
            if not isinstance(d, dict):
                continue
            key = d.get("key", d.get("Key", d.get("k", "")))
            val = d.get("value", d.get("Value", d.get("v", "")))
            key = str(key).strip()
            val = str(val).strip()
            if key == "" and val == "":
                continue
            clean.append({"key": key, "value": val})
        return clean

    def load_cord_gt_kvp(kvp_dir: str, sanity_samples: int = 3) -> Dict[str, List[Dict[str, str]]]:
        gt: Dict[str, List[Dict[str, str]]] = {}

        files = sorted(f for f in os.listdir(kvp_dir) if f.endswith(".json"))
        print(f"[SANITY] KVP_DIR: {kvp_dir}  (files: {len(files)})")

        bad = 0
        shown = 0
        for fname in files:
            doc_id = os.path.splitext(fname)[0]
            fpath = os.path.join(kvp_dir, fname)

            try:
                with open(fpath, "r", encoding="utf-8") as f:
                    data = json.load(f)
            except Exception:
                bad += 1
                continue

            kvps = _coerce_kvp_list(data)
            if kvps is None:
                bad += 1
                continue

            if shown < sanity_samples:
                shown += 1
                raw_head = _read_head(fpath, 200).replace("\n", "\\n")
                print(f"\n[SANITY] GT SAMPLE FILE: {fname}")
                print("[SANITY] RAW_HEAD:", raw_head)
                print("[SANITY] PARSED_KVP_LEN:", len(kvps))

            gt[doc_id] = kvps

        print(f"Loaded KVP ground truth for {len(gt)} CORD documents. Bad files skipped: {bad}")
        if len(gt) == 0:
            raise RuntimeError(
                "Loaded 0 GT documents from KVP_DIR.\n"
                "This usually means you're pointing to the wrong folder.\n"
                f"Use CORD GT: {os.path.join(BASE_DIR,'annotations')}"
            )
        return gt

    # =====================================================
    # 8. LOAD TEXT FILES (CORD)
    # =====================================================

    def load_doc_texts(text_dir: str) -> Dict[str, str]:
        texts: Dict[str, str] = {}
        files = sorted(f for f in os.listdir(text_dir) if f.endswith(".txt"))
        print(f"[SANITY] TEXT_DIR: {text_dir}  (files: {len(files)})")

        for fname in files:
            doc_id = os.path.splitext(fname)[0]
            fpath = os.path.join(text_dir, fname)
            with open(fpath, "r", encoding="utf-8") as f:
                doc_text = f.read()
            texts[doc_id] = doc_text

        return texts

    # =====================================================
    # 9. PROMPT (CORD, GENERIC KVP, FEW-SHOT FROM FILES)
    # =====================================================

    SYSTEM_MESSAGE = (
        "You extract key–value pairs from OCR text of receipts and invoices. "
        "Return ONLY a JSON object with the exact schema "
        "{\"kvp\": [{\"key\":..., \"value\":...}, ...]}. "
        "Do not output any extra text."
    )

    USER_INSTRUCTIONS_BASE = """
    Task: Extract ALL key–value pairs from the given document OCR text.

    Definition:
    - key = the full field label as written in the document (keys may contain spaces/punctuation).
    - value = the associated content for that key.

    CRITICAL extraction rules (to avoid schema drift):
    - DO NOT split multi-word keys into partial keys.
      Example: "SUB TOTAL" must be a single key, NOT {"key":"SUB","value":"TOTAL 17,000"}.
    - Values must NOT include leftover pieces of the key label.
      Example: For key "CHANGE DUE", value should be "3,000" NOT "DUE 3,000".
    - DO NOT use quantities like x1, x2, 2x as keys (they are not field labels).
    - DO NOT use pure numbers/amounts as keys (e.g., "22.000", "63,000.00", "0").
      Numbers/amounts should almost always be VALUES.
    - If a line looks like: ITEM_NAME 18,000
      output: {"key":"ITEM_NAME","value":"18,000"} (keep full item name as the key)

    Output rules (strict):
    - Output ONLY valid JSON (no markdown/code fences, no commentary).
    - JSON must be exactly: {"kvp": [{"key": "...", "value": "..."}, ...]}
    - Include every reasonable key–value pair you can extract.
    - If nothing is extractable: {"kvp": []}

    BAD vs GOOD format examples:
    BAD:  {"kvp":[{"key":"SUB","value":"TOTAL 17,000"}]}
    GOOD: {"kvp":[{"key":"SUB TOTAL","value":"17,000"}]}
    """.strip()

    USER_INSTRUCTIONS_FEWSHOT = (USER_INSTRUCTIONS_BASE + """

    CRITICAL anti-anchoring rules:
    - The examples are ONLY to demonstrate the output format.
    - NEVER copy example values into the new output.
    - For the new receipt, extract pairs ONLY from the text inside <NEW_DOCUMENT>.
    - Ignore all text inside any <EXAMPLE_*> blocks.
    """).strip()

    EXAMPLE_FILES_DIR = os.path.join(BASE_DIR, "cord_prompt_examples")

    EX1_TXT = os.path.join(EXAMPLE_FILES_DIR, "receipt_00730.txt")
    EX1_KVP = os.path.join(EXAMPLE_FILES_DIR, "receipt_00730_kvp.json")
    EX2_TXT = os.path.join(EXAMPLE_FILES_DIR, "receipt_00760.txt")
    EX2_KVP = os.path.join(EXAMPLE_FILES_DIR, "receipt_00760_kvp.json")
    EX3_TXT = os.path.join(EXAMPLE_FILES_DIR, "receipt_00710.txt")
    EX3_KVP = os.path.join(EXAMPLE_FILES_DIR, "receipt_00710_kvp.json")

    def _read_text(path: str) -> str:
        if not os.path.exists(path):
            raise FileNotFoundError(
                f"Missing example file: {path}\n"
                f"Put your CORD example txt+kvp under: {EXAMPLE_FILES_DIR}"
            )
        with open(path, "r", encoding="utf-8") as f:
            return f.read().strip()

    def _read_kvp_json_as_prompt_obj(path: str) -> str:
        raw = _read_text(path)
        data = json.loads(raw)
        if isinstance(data, dict) and "kvp" in data:
            obj = data
        elif isinstance(data, list):
            obj = {"kvp": data}
        else:
            raise ValueError(f"Unexpected example KVP format in {path}: expected list or {{'kvp':[...]}}")
        return json.dumps(obj, ensure_ascii=False, indent=2)

    def load_cord_examples() -> Dict[str, Dict[str, str]]:
        ex: Dict[str, Dict[str, str]] = {}
        ex["ex1"] = {"doc": _read_text(EX1_TXT), "json": _read_kvp_json_as_prompt_obj(EX1_KVP)}
        ex["ex2"] = {"doc": _read_text(EX2_TXT), "json": _read_kvp_json_as_prompt_obj(EX2_KVP)}
        ex["ex3"] = {"doc": _read_text(EX3_TXT), "json": _read_kvp_json_as_prompt_obj(EX3_KVP)}
        return ex

    CORD_EX: Optional[Dict[str, Dict[str, str]]] = None

    def _ensure_examples_loaded():
        nonlocal CORD_EX
        if CORD_EX is None:
            CORD_EX = load_cord_examples()

    def _example_block(idx: int, doc_text: str, out_json: str) -> str:
        return (
            f"<EXAMPLE_{idx}_DOCUMENT>\n{doc_text}\n</EXAMPLE_{idx}_DOCUMENT>\n"
            f"<EXAMPLE_{idx}_OUTPUT>\n{out_json}\n</EXAMPLE_{idx}_OUTPUT>\n"
        )

    def _task_footer(doc_text: str) -> str:
        return (
            "Now do the task for the NEW document below.\n"
            "Extract pairs ONLY from <NEW_DOCUMENT>. Ignore all <EXAMPLE_*> blocks completely.\n"
            "Return ONLY a SINGLE JSON object and stop immediately after the final '}'.\n"
            "<NEW_DOCUMENT>\n"
            f"{doc_text}\n"
            "</NEW_DOCUMENT>\n"
            "Output:\n"
        )

    def build_prompt_0shot(doc_text: str) -> str:
        prompt = USER_INSTRUCTIONS_BASE + "\n\n" + _task_footer(doc_text)
        assert "<EXAMPLE_1_DOCUMENT>" not in prompt, "0-shot prompt accidentally contains example blocks."
        return prompt

    def build_prompt_1shot(doc_text: str) -> str:
        _ensure_examples_loaded()
        ex1 = CORD_EX["ex1"]
        return "\n\n".join([
            USER_INSTRUCTIONS_FEWSHOT,
            _example_block(1, ex1["doc"], ex1["json"]),
            _task_footer(doc_text),
        ])

    def build_prompt_2shot(doc_text: str) -> str:
        _ensure_examples_loaded()
        ex1 = CORD_EX["ex1"]
        ex2 = CORD_EX["ex2"]
        return "\n\n".join([
            USER_INSTRUCTIONS_FEWSHOT,
            _example_block(1, ex1["doc"], ex1["json"]),
            _example_block(2, ex2["doc"], ex2["json"]),
            _task_footer(doc_text),
        ])

    def build_prompt_3shot(doc_text: str) -> str:
        _ensure_examples_loaded()
        ex1 = CORD_EX["ex1"]
        ex2 = CORD_EX["ex2"]
        ex3 = CORD_EX["ex3"]
        return "\n\n".join([
            USER_INSTRUCTIONS_FEWSHOT,
            _example_block(1, ex1["doc"], ex1["json"]),
            _example_block(2, ex2["doc"], ex2["json"]),
            _example_block(3, ex3["doc"], ex3["json"]),
            _task_footer(doc_text),
        ])

    def build_prompt(doc_text: str) -> str:
        if SHOT_VARIANT == "0shot":
            return build_prompt_0shot(doc_text)
        if SHOT_VARIANT == "1shot":
            return build_prompt_1shot(doc_text)
        if SHOT_VARIANT == "2shot":
            return build_prompt_2shot(doc_text)
        if SHOT_VARIANT == "3shot":
            return build_prompt_3shot(doc_text)
        raise ValueError(f"Unknown SHOT_VARIANT: {SHOT_VARIANT}")

    # =====================================================
    # 10. PARSING
    # =====================================================

    def parse_kvp_output(raw_output: str) -> List[Dict[str, str]]:
        """
        Parse LLM output conservatively while preserving complete KVP objects.

        1. Prefer a complete valid JSON response:
           {"kvp": [{"key": "...", "value": "..."}, ...]}
        2. If the full response is malformed/truncated, recover only individually
           COMPLETE, valid JSON objects already present in the generated text
           that contain string-valued "key" and "value" fields.

        No regex KVP extraction, malformed-JSON repair, guessing, fuzzy parsing,
        GT-aware recovery, or document-text fallback is performed.
        """
        if not isinstance(raw_output, str):
            return []

        s = raw_output.strip()
        if not s:
            return []

        if s.startswith("```"):
            lines = s.splitlines()
            if lines and lines[0].strip().startswith("```"):
                lines = lines[1:]
            if lines and lines[-1].strip() == "```":
                lines = lines[:-1]
            s = "\n".join(lines).strip()

        try:
            obj = json.loads(s)
        except json.JSONDecodeError:
            obj = None

        if isinstance(obj, dict):
            kvp_list = obj.get("kvp")
            if isinstance(kvp_list, list):
                parsed = []
                for pair in kvp_list:
                    if not isinstance(pair, dict):
                        continue
                    key = pair.get("key")
                    value = pair.get("value")
                    if not isinstance(key, str) or not isinstance(value, str):
                        continue
                    key = key.strip()
                    value = value.strip()
                    if key == "" and value == "":
                        continue
                    parsed.append({"key": key, "value": value})
                return parsed

        decoder = json.JSONDecoder()
        recovered = []

        for start, char in enumerate(s):
            if char != "{":
                continue
            try:
                candidate, _ = decoder.raw_decode(s[start:])
            except json.JSONDecodeError:
                continue

            if not isinstance(candidate, dict):
                continue

            key = candidate.get("key")
            value = candidate.get("value")
            if not isinstance(key, str) or not isinstance(value, str):
                continue

            key = key.strip()
            value = value.strip()
            if key == "" and value == "":
                continue

            recovered.append({"key": key, "value": value})

        return recovered

    # =====================================================
    # 11. METRICS
    # =====================================================

    # NOTE ON METRICS:
    # - Key Recall is duplicate-aware and uses formatting-tolerant normalized keys.
    # - Exact Match requires the normalized key and normalized value to match exactly.
    # - Value F1 is token-overlap F1 on values, matched by normalized key with no prediction reuse.
    #   Every GT pair contributes to the final average; missing keys and zero-overlap values receive 0.
    #   This implementation is intentionally aligned with the manuscript definition.

    def compute_doc_metrics(
        gt_kvp: List[Dict[str, str]],
        pred_kvp: List[Dict[str, str]]
    ) -> Dict[str, Any]:

        num_gt = len(gt_kvp)

        if num_gt == 0:
            return {
                "num_gt_keys": 0,
                "matched_keys": 0,
                "key_recall": 0.0,
                "exact_matches": 0,
                "exact_match_rate": 0.0,
                "value_f1": 0.0,
            }

        # -------------------------------------------------
        # Normalize GT and predicted key/value strings
        # -------------------------------------------------
        gt_keys_norm = [
            normalize_key(kv.get("key", ""))
            for kv in gt_kvp
        ]
        gt_vals_norm = [
            normalize_str(kv.get("value", ""))
            for kv in gt_kvp
        ]

        pred_keys_norm = [
            normalize_key(kv.get("key", ""))
            for kv in pred_kvp
        ]
        pred_vals_norm = [
            normalize_str(kv.get("value", ""))
            for kv in pred_kvp
        ]

        # =================================================
        # 1. KEY RECALL
        # Duplicate-aware key matching
        # =================================================

        gt_key_counts = Counter(k for k in gt_keys_norm if k)
        pred_key_counts = Counter(k for k in pred_keys_norm if k)

        matched_keys = sum(
            min(gt_key_counts[k], pred_key_counts.get(k, 0))
            for k in gt_key_counts
        )

        key_recall = matched_keys / num_gt

        # =================================================
        # 2. EXACT MATCH RATE
        # Exact normalized key AND exact normalized value.
        # Each prediction can be used at most once.
        # =================================================

        used_pred_exact = set()
        exact_matches = 0

        for gt_key, gt_value in zip(gt_keys_norm, gt_vals_norm):

            if not gt_key or not gt_value:
                continue

            for j, (pred_key, pred_value) in enumerate(
                zip(pred_keys_norm, pred_vals_norm)
            ):
                if j in used_pred_exact:
                    continue

                if gt_key == pred_key and gt_value == pred_value:
                    exact_matches += 1
                    used_pred_exact.add(j)
                    break

        exact_match_rate = exact_matches / num_gt

        # =================================================
        # 3. VALUE TOKEN F1
        #
        # For every GT pair:
        #   - consider unused predictions with the same key
        #   - select the one with maximum token-level F1
        #   - same-key but zero-overlap value -> F1 = 0
        #   - no same-key prediction -> F1 = 0
        #
        # Final Value F1 is averaged over ALL GT pairs.
        # =================================================

        used_pred_f1 = set()
        value_f1_scores = []

        for gt_pair in gt_kvp:

            gt_key = normalize_key(gt_pair.get("key", ""))
            gt_value = str(gt_pair.get("value", ""))

            # A malformed/empty GT field receives zero credit.
            if not gt_key or not normalize_str(gt_value):
                value_f1_scores.append(0.0)
                continue

            best_j = None
            best_f1 = 0.0

            for j, pred_pair in enumerate(pred_kvp):

                if j in used_pred_f1:
                    continue

                pred_key = normalize_key(pred_pair.get("key", ""))

                if pred_key != gt_key or not pred_key:
                    continue

                pred_value = str(pred_pair.get("value", ""))

                f1 = text_f1(gt_value, pred_value)

                # Important:
                # best_j must also be assigned when F1 == 0.
                if best_j is None or f1 > best_f1:
                    best_j = j
                    best_f1 = f1

            if best_j is not None:
                used_pred_f1.add(best_j)
                value_f1_scores.append(best_f1)
            else:
                # Missing key
                value_f1_scores.append(0.0)

        value_f1 = float(np.mean(value_f1_scores))

        return {
            "num_gt_keys": num_gt,
            "matched_keys": int(matched_keys),
            "key_recall": float(key_recall),
            "exact_matches": int(exact_matches),
            "exact_match_rate": float(exact_match_rate),
            "value_f1": float(value_f1),
        }

    # =====================================================
    # 12. MODEL LOADING + GENERATION (NO BNB)
    # =====================================================

    def _preferred_dtype() -> torch.dtype:
        if DEVICE == "cuda":
            return torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
        return torch.float32

    def build_model_and_tokenizer(model_name: str, family: str):
        print(f"Loading model: {model_name}  (family={family})")

        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()

        kwargs = {}
        if HF_TOKEN:
            kwargs["token"] = HF_TOKEN

        config = AutoConfig.from_pretrained(model_name, **kwargs)
        tokenizer = AutoTokenizer.from_pretrained(model_name, **kwargs)

        is_encoder_decoder = bool(getattr(config, "is_encoder_decoder", False)) or (family == "seq2seq")

        if not is_encoder_decoder:
            tokenizer.padding_side = "left"

        dtype = _preferred_dtype()
        primary_device_map = "cuda" if DEVICE == "cuda" else None

        def _load(device_map):
            if is_encoder_decoder:
                return AutoModelForSeq2SeqLM.from_pretrained(
                    model_name,
                    device_map=device_map,
                    dtype=dtype,
                    **kwargs,
                )
            return AutoModelForCausalLM.from_pretrained(
                model_name,
                device_map=device_map,
                dtype=dtype,
                **kwargs,
            )

        try:
            model = _load(primary_device_map)
        except Exception as e:
            # Do not silently change device placement. A failed CUDA load must be
            # surfaced so the run cannot continue under a different hardware
            # configuration (for example CPU offloading via device_map="auto").
            raise RuntimeError(
                f"Failed to load {model_name} with device_map={primary_device_map!r}. "
                "Check local disk space/model cache and GPU memory before retrying."
            ) from e

        if tokenizer.pad_token is None and tokenizer.eos_token is not None:
            tokenizer.pad_token = tokenizer.eos_token
            model.config.pad_token_id = tokenizer.eos_token_id

        print("hf_device_map:", getattr(model, "hf_device_map", None))
        model.eval()
        return tokenizer, model, is_encoder_decoder

    def _effective_max_input_tokens(tokenizer) -> int:
        mml = getattr(tokenizer, "model_max_length", None)
        if mml is None or mml > 100000:
            return MAX_INPUT_TOKENS
        return int(min(MAX_INPUT_TOKENS, mml))

    def _maybe_apply_chat_template(tokenizer, user_prompt: str) -> str:
        try:
            tmpl = getattr(tokenizer, "chat_template", None)
            if tmpl:
                messages = [
                    {"role": "system", "content": SYSTEM_MESSAGE},
                    {"role": "user", "content": user_prompt},
                ]
                return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        except Exception:
            pass
        return user_prompt

    def _qwen_im_end_eos_id(tokenizer) -> Optional[int]:
        try:
            im_end_id = tokenizer.convert_tokens_to_ids("<|im_end|>")
            if im_end_id is not None and im_end_id != tokenizer.unk_token_id:
                return int(im_end_id)
        except Exception:
            pass
        return None

    def _postprocess_decoded_text(s: str) -> str:
        s = s.strip()
        s = s.replace("<|im_end|>", "").strip()
        return s

    def run_model_on_prompts(tokenizer, model, prompts: List[str], is_encoder_decoder: bool):
        """Generate outputs plus neutral generation-length diagnostics."""
        if not is_encoder_decoder:
            prompts = [_maybe_apply_chat_template(tokenizer, p) for p in prompts]

        max_in = _effective_max_input_tokens(tokenizer)

        inputs = tokenizer(
            prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=max_in,
        )

        if DEVICE == "cuda":
            inputs = {k: v.to("cuda") for k, v in inputs.items()}

        eos_id = _qwen_im_end_eos_id(tokenizer) or int(tokenizer.eos_token_id)

        gen_kwargs = dict(
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=int(tokenizer.pad_token_id),
            eos_token_id=int(eos_id),
            return_dict_in_generate=True,
            output_scores=False,
        )

        with torch.no_grad():
            gen_out = model.generate(**inputs, **gen_kwargs)

        sequences = gen_out.sequences
        results = []

        if is_encoder_decoder:
            for i in range(sequences.shape[0]):
                ids = sequences[i]
                generated_tokens = int(ids.shape[-1])
                text = tokenizer.decode(ids, skip_special_tokens=True).strip()
                results.append({
                    "text": text,
                    "generated_tokens": generated_tokens,
                    "output_hit_max_new_tokens": bool(generated_tokens >= MAX_NEW_TOKENS),
                })
            return results

        input_width = int(inputs["input_ids"].shape[1])

        for i in range(sequences.shape[0]):
            ids = sequences[i, input_width:]
            generated_tokens = int(ids.shape[-1])
            text = tokenizer.decode(ids, skip_special_tokens=False)
            text = _postprocess_decoded_text(text)

            results.append({
                "text": text,
                "generated_tokens": generated_tokens,
                "output_hit_max_new_tokens": bool(generated_tokens >= MAX_NEW_TOKENS),
            })

        return results

    # =====================================================
    # 13. PROMPT TOKEN LOGGING
    # =====================================================

    def prompt_token_stats(tokenizer, prompt: str) -> Tuple[int, int, bool]:
        max_in = _effective_max_input_tokens(tokenizer)
        tok_no_trunc = tokenizer(prompt, return_tensors="pt", truncation=False)
        tok_trunc    = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=max_in)
        n_no_trunc = int(tok_no_trunc["input_ids"].shape[-1])
        n_trunc    = int(tok_trunc["input_ids"].shape[-1])
        return n_no_trunc, n_trunc, (n_no_trunc > n_trunc)

    # =====================================================
    # 14. MAIN EVAL LOOP (CORD)
    # =====================================================

    def _cleanup_cuda():
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()

    def evaluate_model_on_cord(model_name: str, family: str):
        tokenizer = None
        model = None
        is_encoder_decoder = False

        try:
            print("\n" + "=" * 60)
            print(f"=== Running model {model_name} on CORD ({ENGINE}, {SHOT_VARIANT}) ===")

            gt_dict   = load_cord_gt_kvp(KVP_DIR)
            text_dict = load_doc_texts(TEXT_DIR)

            gt_ids = set(gt_dict.keys())
            txt_ids = set(text_dict.keys())
            inter = sorted(gt_ids & txt_ids)

            print(f"[SANITY] GT docs:   {len(gt_ids)}")
            print(f"[SANITY] TXT docs:  {len(txt_ids)}")
            print(f"[SANITY] INTERSECT: {len(inter)}")
            print(f"[SANITY] GT-only:   {len(gt_ids - txt_ids)}")
            print(f"[SANITY] TXT-only:  {len(txt_ids - gt_ids)}")

            if len(inter) == 0:
                raise RuntimeError(
                    "No intersecting doc_ids between GT and text.\n"
                    "Check that filenames match (receipt_XXXXX) and you picked the right TEXT_DIR / KVP_DIR."
                )

            empty_gt = sum(1 for d in inter if len(gt_dict.get(d, [])) == 0)
            print(f"[SANITY] Empty GT kvp among intersected docs: {empty_gt}/{len(inter)}")
            if empty_gt == len(inter):
                raise RuntimeError(
                    "All GT kvp lists are empty for intersected docs.\n"
                    "This happens if you pointed to the wrong GT folder (e.g., cord_kvp placeholder) "
                    "or the schema mapping is wrong.\n"
                    f"Use CORD GT: {os.path.join(BASE_DIR,'annotations')}"
                )

            doc_ids = inter
            if MAX_DOCS is not None:
                doc_ids = doc_ids[:MAX_DOCS]
            print(f"Total docs to run: {len(doc_ids)}")

            out_dir_name = f"cord_{ENGINE}_{SHOT_VARIANT}_{model_name.replace('/', '_')}"
            OUTPUT_DIR   = os.path.join(RESULTS_BASE, out_dir_name)
            os.makedirs(OUTPUT_DIR, exist_ok=True)
            print(f"Results will be saved under: {OUTPUT_DIR}")

            csv_path  = os.path.join(OUTPUT_DIR, f"metrics_{ENGINE}_{SHOT_VARIANT}_{model_name.replace('/', '_')}.csv")
            pred_path = os.path.join(OUTPUT_DIR, f"predictions_{ENGINE}_{SHOT_VARIANT}_{model_name.replace('/', '_')}.jsonl")

            if not RESUME:
                if os.path.exists(csv_path):
                    os.remove(csv_path)
                if os.path.exists(pred_path):
                    os.remove(pred_path)

            processed_ids = set()
            csv_exists = os.path.exists(csv_path)
            if RESUME and csv_exists:
                with open(csv_path, "r", encoding="utf-8") as f:
                    lines = f.readlines()
                if len(lines) > 1:
                    for line in lines[1:]:
                        parts = line.strip().split(",")
                        if parts and parts[0]:
                            processed_ids.add(parts[0])
                print(f"Found existing metrics CSV with {len(processed_ids)} docs. Resuming from there.")

            doc_ids_to_run = [d for d in doc_ids if d not in processed_ids]
            print(f"Docs remaining for this model: {len(doc_ids_to_run)}")

            if len(doc_ids_to_run) != 0:
                tokenizer, model, is_encoder_decoder = build_model_and_tokenizer(model_name, family)

                csv_mode = "a" if (RESUME and csv_exists) else "w"
                with open(csv_path, csv_mode, encoding="utf-8") as csv_f, \
                     open(pred_path, "a", encoding="utf-8") as pred_f:

                    if csv_mode == "w":
                        csv_f.write(
                            "doc_id,num_gt_keys,matched_keys,key_recall,exact_match_rate,value_f1,"
                            "prompt_tokens_no_trunc,prompt_tokens_trunc,prompt_truncated,"
                            "generated_tokens,output_hit_max_new_tokens,"
                            "pred_num_pairs,pred_unique_keys,pred_dup_key_frac,pred_single_token_key_frac,pred_numeric_key_frac,pred_qty_key_frac\n"
                        )

                    with tqdm(
                        total=len(doc_ids_to_run),
                        desc=f"{model_name} [{ENGINE}][{SHOT_VARIANT}]",
                        dynamic_ncols=True
                    ) as pbar:

                        for start in range(0, len(doc_ids_to_run), BATCH_SIZE):
                            batch_ids    = doc_ids_to_run[start:start + BATCH_SIZE]
                            batch_texts  = [text_dict[d] for d in batch_ids]
                            batch_gt_kvp = [gt_dict[d] for d in batch_ids]

                            batch_prompts = [build_prompt(t) for t in batch_texts]
                            batch_prompt_stats = [prompt_token_stats(tokenizer, p) for p in batch_prompts]

                            if DEBUG_FIRST_DOC and start == 0:
                                p0 = batch_prompts[0]
                                max_in = _effective_max_input_tokens(tokenizer)
                                n_no_trunc, n_trunc, truncated = batch_prompt_stats[0]
                                print("\n===== DEBUG: PROMPT TOKEN ANALYSIS =====")
                                print(f"Tokenizer model_max_length: {getattr(tokenizer, 'model_max_length', None)}")
                                print(f"Effective max_input_tokens: {max_in}")
                                print(f"Prompt tokens (no trunc):   {n_no_trunc}")
                                print(f"Prompt tokens (truncated):  {n_trunc}")
                                if truncated:
                                    print("⚠️ Prompt was truncated.")
                                print("Contains <NEW_DOCUMENT> footer:", "<NEW_DOCUMENT>" in p0 and "</NEW_DOCUMENT>" in p0)
                                print("\n----- PROMPT TAIL (last 1200 chars) -----")
                                print(p0[-1200:])
                                print("===== END DEBUG =====\n")

                            generation_results = run_model_on_prompts(tokenizer, model, batch_prompts, is_encoder_decoder)

                            for (doc_id, doc_text, gt_kvp, gen_result, pstats) in zip(
                                batch_ids, batch_texts, batch_gt_kvp, generation_results, batch_prompt_stats
                            ):
                                n_no_trunc, n_trunc, truncated = pstats
                                raw_output = gen_result["text"]
                                generated_tokens = int(gen_result["generated_tokens"])
                                output_hit_max_new_tokens = bool(gen_result["output_hit_max_new_tokens"])

                                pred_kvp = parse_kvp_output(raw_output)
                                metrics  = compute_doc_metrics(gt_kvp, pred_kvp)
                                drift    = compute_pred_drift_stats(pred_kvp)

                                csv_f.write(
                                    f"{doc_id},{metrics['num_gt_keys']},{metrics['matched_keys']},"
                                    f"{metrics['key_recall']:.10f},{metrics['exact_match_rate']:.10f},"
                                    f"{metrics['value_f1']:.10f},{n_no_trunc},{n_trunc},{int(truncated)},"
                                    f"{generated_tokens},{int(output_hit_max_new_tokens)},"
                                    f"{drift['pred_num_pairs']},{drift['pred_unique_keys']},"
                                    f"{drift['pred_dup_key_frac']:.4f},{drift['pred_single_token_key_frac']:.4f},"
                                    f"{drift['pred_numeric_key_frac']:.4f},{drift['pred_qty_key_frac']:.4f}\n"
                                )
                                csv_f.flush()
                                os.fsync(csv_f.fileno())

                                record = {
                                    "doc_id": doc_id,
                                    "engine": ENGINE,
                                    "shot_variant": SHOT_VARIANT,
                                    "model": model_name,
                                    "gt_kvp": gt_kvp,
                                    "pred_kvp": pred_kvp,
                                    "raw_output": raw_output,
                                    "metrics": metrics,
                                    "drift": drift,
                                    "prompt_tokens_no_trunc": n_no_trunc,
                                    "prompt_tokens_trunc": n_trunc,
                                    "prompt_truncated": bool(truncated),
                                    "generated_tokens": generated_tokens,
                                    "output_hit_max_new_tokens": output_hit_max_new_tokens,
                                    "seed": SEED,
                                    "dtype": str(_preferred_dtype()),
                                }
                                pred_f.write(json.dumps(record, ensure_ascii=False) + "\n")
                                pred_f.flush()
                                os.fsync(pred_f.fileno())

                                pbar.update(1)

            macro_key_recall = 0.0
            macro_em         = 0.0
            macro_value_f1   = 0.0
            n_docs           = 0

            if os.path.exists(csv_path):
                with open(csv_path, "r", encoding="utf-8") as f:
                    lines = f.readlines()
                if len(lines) > 1:
                    sum_key_rec = 0.0
                    sum_em      = 0.0
                    sum_f1      = 0.0
                    for line in lines[1:]:
                        parts = line.strip().split(",")
                        if len(parts) < 6:
                            continue
                        try:
                            key_rec = float(parts[3])
                            em      = float(parts[4])
                            f1      = float(parts[5])
                        except ValueError:
                            continue
                        sum_key_rec += key_rec
                        sum_em      += em
                        sum_f1      += f1
                        n_docs      += 1
                    if n_docs > 0:
                        macro_key_recall = sum_key_rec / n_docs
                        macro_em         = sum_em / n_docs
                        macro_value_f1   = sum_f1 / n_docs

            print("\n==== SUMMARY (MACRO AVERAGE OVER DOCS) ====")
            print("Dataset: CORD")
            print(f"Engine:  {ENGINE}")
            print(f"Shot:    {SHOT_VARIANT}")
            print(f"Model:   {model_name}")
            print(f"Docs (from CSV): {n_docs}")
            print(f"Key Recall: {macro_key_recall:.4f}")
            print(f"EM:         {macro_em:.4f}")
            print(f"Value F1:   {macro_value_f1:.4f}")
            print(f"\nPer-doc metrics saved to: {csv_path}")
            print(f"Predictions saved to:      {pred_path}")
            print("=" * 60 + "\n")

        finally:
            try:
                if model is not None:
                    del model
                if tokenizer is not None:
                    del tokenizer
            except Exception:
                pass
            _cleanup_cuda()

    # =====================================================
    # 15. MAIN
    # =====================================================

    def main():
        for spec in tqdm(MODEL_SPECS, desc=f"Models [{SHOT_VARIANT}]", dynamic_ncols=True):
            evaluate_model_on_cord(
                model_name=spec["name"],
                family=spec["family"],
            )

    # The runner is intentionally not auto-executed here.
    # Use the experiment cells later in the notebook.

    def configure(engine, shot, model_specs):
        """Configure one CORD experiment without modifying any other dataset."""
        nonlocal ENGINE, SHOT_VARIANT, TEXT_DIR, MODEL_SPECS, RESUME, RESULTS_BASE

        valid_engines = {"gold_text", "tesseract", "paddleocr", "easyocr"}
        valid_shots = {"0shot", "1shot", "2shot", "3shot"}

        if engine not in valid_engines:
            raise ValueError(f"Unsupported engine: {engine}")
        if shot not in valid_shots:
            raise ValueError(f"Unsupported shot setting: {shot}")

        ENGINE = engine
        SHOT_VARIANT = shot
        MODEL_SPECS = list(model_specs)
        RESUME = True
        RESULTS_BASE = "/content/drive/MyDrive/data/pixels_to_pairs_final_results/cord"
        os.makedirs(RESULTS_BASE, exist_ok=True)

        text_paths = {
            "gold_text": os.path.join(BASE_DIR, "cord_gold_text"),
            "tesseract": os.path.join(BASE_DIR, "cord_ocr", "tesseract"),
            "paddleocr": os.path.join(BASE_DIR, "cord_ocr", "paddleocr"),
            "easyocr": os.path.join(BASE_DIR, "cord_ocr", "easyocr"),
        }
        TEXT_DIR = text_paths[engine]

        if "/pixels_to_pairs_final_results/cord" not in RESULTS_BASE:
            raise RuntimeError(f"CORD results path is incorrect: {RESULTS_BASE}")
        if "/data/cord/" not in TEXT_DIR:
            raise RuntimeError(f"CORD text path is incorrect: {TEXT_DIR}")

        print("\nCONFIG CHECK — CORD")
        print("  ENGINE:", ENGINE)
        print("  SHOT:", SHOT_VARIANT)
        print("  TEXT_DIR:", TEXT_DIR)
        print("  RESULTS_BASE:", RESULTS_BASE)
        print("  MAX_NEW_TOKENS:", MAX_NEW_TOKENS)
        print("  MODELS:", [m["name"] for m in MODEL_SPECS])

    def run(engine, shot, model_specs):
        """Run one complete CORD condition sequentially."""
        configure(engine, shot, model_specs)
        for spec in MODEL_SPECS:
            evaluate_model_on_cord(spec["name"], spec["family"])

    def status():
        return {
            "dataset": "CORD",
            "engine": ENGINE,
            "shot": SHOT_VARIANT,
            "text_dir": TEXT_DIR,
            "results_base": RESULTS_BASE,
            "max_new_tokens": MAX_NEW_TOKENS,
            "resume": RESUME,
        }

    return SimpleNamespace(
        configure=configure,
        run=run,
        status=status,
        evaluate=evaluate_model_on_cord,
    )

CORD_RUNNER = build_cord_runner()
print("CORD runner ready:", CORD_RUNNER.status())


## 6. Experiment controller

In [ ]:
# ============================================================
# BENCHMARK MODEL SETS + DATASET RUNNER WRAPPERS
# ============================================================
# This cell DEFINES configuration only. It does not execute experiments.

ALL_MODELS = [
    {"name": "Qwen/Qwen2.5-7B-Instruct", "family": "causal"},
    {"name": "meta-llama/Meta-Llama-3-8B-Instruct", "family": "causal"},
    {"name": "mistralai/Mistral-7B-Instruct-v0.2", "family": "causal"},
    {"name": "google/gemma-2b-it", "family": "causal"},
    {"name": "google/gemma-7b-it", "family": "causal"},
]

FEWSHOT_MODELS = [
    {"name": "Qwen/Qwen2.5-7B-Instruct", "family": "causal"},
    {"name": "meta-llama/Meta-Llama-3-8B-Instruct", "family": "causal"},
]


def run_funsd_experiment(engine, shot, model_specs):
    FUNSD_RUNNER.run(engine, shot, model_specs)


def run_sroie_experiment(engine, shot, model_specs):
    SROIE_RUNNER.run(engine, shot, model_specs)


def run_cord_experiment(engine, shot, model_specs):
    CORD_RUNNER.run(engine, shot, model_specs)


print("Benchmark model sets and dataset runner wrappers ready.")
print("No experiments are executed by this cell.")


## 7. Pre-flight validation

In [ ]:
import os

checks = {
    "FUNSD": [
        "/content/drive/MyDrive/data/funsd/funsd_gold_text_spatial",
        "/content/drive/MyDrive/data/funsd/funsd_ocr/tesseract",
        "/content/drive/MyDrive/data/funsd/funsd_ocr/paddleocr",
        "/content/drive/MyDrive/data/funsd/funsd_ocr/easyocr",
    ],
    "SROIE": [
        "/content/drive/MyDrive/data/sroie/sroie_gold_text",
        "/content/drive/MyDrive/data/sroie/sroie_ocr/tesseract",
        "/content/drive/MyDrive/data/sroie/sroie_ocr/paddleocr",
        "/content/drive/MyDrive/data/sroie/sroie_ocr/easyocr",
    ],
    "CORD": [
        "/content/drive/MyDrive/data/cord/cord_gold_text",
        "/content/drive/MyDrive/data/cord/cord_ocr/tesseract",
        "/content/drive/MyDrive/data/cord/cord_ocr/paddleocr",
        "/content/drive/MyDrive/data/cord/cord_ocr/easyocr",
    ],
}

runners = {
    "FUNSD": FUNSD_RUNNER,
    "SROIE": SROIE_RUNNER,
    "CORD": CORD_RUNNER,
}

all_ok = True

for dataset, paths in checks.items():
    print(f"\n{dataset}")
    st = runners[dataset].status()

    expected = f"/pixels_to_pairs_final_results/{dataset.lower()}"
    out_ok = expected in st["results_base"]
    print("  OUTPUT:", "OK" if out_ok else "FAIL", st["results_base"])
    all_ok &= out_ok

    for p in paths:
        ok = os.path.isdir(p)
        print(" ", "OK" if ok else "MISSING", p)
        all_ok &= ok

assert all_ok, "Pre-flight failed. Fix any missing input directory before starting the final sweep."

print("\nPASS: input paths exist and FUNSD/SROIE/CORD output trees are isolated.")


## 8. Disk-safe cache management

Large Hugging Face checkpoints are cached on the Colab VM under
`/root/.cache/huggingface/hub`. Running several 7–8B models can fill the
ephemeral Colab disk even though benchmark outputs are stored safely on
Google Drive.

The helpers below **never touch Google Drive or benchmark result folders**.
They only remove local Hugging Face model caches after a model has finished
all of its required experiments.

The final benchmark protocol is unchanged: prompts, decoding, 1024-token
generation cap, parsing, key normalization, zero-GT exclusion, and metrics
remain exactly the same.

In [ ]:
import gc
import os
import shutil
import torch

HF_HUB_CACHE = "/root/.cache/huggingface/hub"
MIN_FREE_DISK_GB = 22.0  # headroom for large checkpoint download/reconstruction

def _free_disk_gb(path="/"):
    return shutil.disk_usage(path).free / (1024 ** 3)

def show_disk_status():
    free_gb = _free_disk_gb("/")
    print(f"Free local disk: {free_gb:.1f} GB")
    os.system("du -sh /root/.cache/huggingface 2>/dev/null || true")
    return free_gb

def hf_model_cache_path(model_name):
    return os.path.join(
        HF_HUB_CACHE,
        "models--" + model_name.replace("/", "--")
    )

def release_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        try:
            torch.cuda.ipc_collect()
        except Exception:
            pass

def remove_model_cache(model_name):
    """
    Remove only this model's LOCAL Hugging Face cache.
    Google Drive datasets/results are untouched.
    """
    release_memory()
    path = hf_model_cache_path(model_name)
    if os.path.exists(path):
        print(f"Removing local cache: {path}")
        shutil.rmtree(path)
    else:
        print(f"No local cache to remove for: {model_name}")
    release_memory()
    print(f"Free local disk after cleanup: {_free_disk_gb('/'):.1f} GB")


def ensure_disk_headroom(current_model=None, min_free_gb=MIN_FREE_DISK_GB):
    """
    Ensure enough local disk exists before a large model download/load.

    If disk is low, remove benchmark-model caches left from earlier models.
    The cache for `current_model` is preserved unless it is itself an
    incomplete cache that the caller explicitly removes.
    """
    free_gb = _free_disk_gb("/")
    if free_gb >= min_free_gb:
        print(f"Disk OK: {free_gb:.1f} GB free.")
        return

    print(
        f"Low local disk: {free_gb:.1f} GB free; "
        f"target is >= {min_free_gb:.1f} GB."
    )

    for spec in ALL_MODELS:
        name = spec["name"]
        if name == current_model:
            continue
        path = hf_model_cache_path(name)
        if os.path.exists(path):
            print(f"  deleting stale cache: {name}")
            shutil.rmtree(path)
            release_memory()
            free_gb = _free_disk_gb("/")
            if free_gb >= min_free_gb:
                break

    print(f"Free local disk now: {free_gb:.1f} GB")
    if free_gb < min_free_gb:
        raise RuntimeError(
            f"Only {free_gb:.1f} GB local disk is free after safe cache cleanup. "
            "Do not start another large-model run until more Colab VM disk is freed."
        )

print("Disk/cache helpers ready.")
show_disk_status()

### One-time local-cache cleanup before the master run

Run the next cell once after a disk-full warning. It removes only local Hugging Face benchmark-model caches. Saved datasets, predictions, and metrics on Google Drive are untouched.


In [ ]:
# Safe local-cache cleanup after a Colab disk-full warning.
# This NEVER deletes Google Drive benchmark results.
for _spec in ALL_MODELS:
    _path = hf_model_cache_path(_spec["name"])
    if os.path.exists(_path):
        print(f"Removing local model cache: {_spec['name']}")
        shutil.rmtree(_path)
release_memory()
show_disk_status()
ensure_disk_headroom(current_model=None, min_free_gb=MIN_FREE_DISK_GB)


## 9. Disk-safe resumable experiment queue

The queue is now **model-major**: one model is downloaded, all benchmark
conditions assigned to that model are run, and only then is its local
checkpoint cache removed. This prevents six large checkpoints from
accumulating on the Colab VM.

`RESUME=True` inside the dataset runners remains authoritative, so completed
Drive outputs are skipped/resumed rather than recomputed. This makes the
same queue suitable both for a fresh full run and for continuing the current
partially completed benchmark.

Qwen and LLaMA receive the predefined few-shot conditions. All six models
receive the 0-shot conditions, exactly as in the original experiment matrix.

In [ ]:
# ============================================================
# AUTHORITATIVE BENCHMARK SCHEDULER DEFINITION
# ============================================================
# This cell defines the complete experiment matrix and the model-major
# scheduler. It does NOT start the benchmark.
#
# Experiment policy:
#   - All 5 valid models: 0-shot on Gold, PaddleOCR, EasyOCR, Tesseract
#   - Qwen + LLaMA: 1/2/3-shot on Gold + PaddleOCR
#   - Same policy for FUNSD, SROIE, and CORD
#
# The ONLY cell that actually starts experiments is the final
# "RUN BENCHMARK — ONLY EXECUTION CELL" below.

FUNSD_ZERO_SHOT_CONDITIONS = [
    ("gold_text_spatial", "0shot"),
    ("paddleocr", "0shot"),
    ("easyocr", "0shot"),
    ("tesseract", "0shot"),
]

FUNSD_FEWSHOT_CONDITIONS = [
    ("gold_text_spatial", "1shot"),
    ("gold_text_spatial", "2shot"),
    ("gold_text_spatial", "3shot"),
    ("paddleocr", "1shot"),
    ("paddleocr", "2shot"),
    ("paddleocr", "3shot"),
]

SROIE_ZERO_SHOT_CONDITIONS = [
    ("gold_text", "0shot"),
    ("paddleocr", "0shot"),
    ("easyocr", "0shot"),
    ("tesseract", "0shot"),
]

SROIE_FEWSHOT_CONDITIONS = [
    ("gold_text", "1shot"),
    ("gold_text", "2shot"),
    ("gold_text", "3shot"),
    ("paddleocr", "1shot"),
    ("paddleocr", "2shot"),
    ("paddleocr", "3shot"),
]

CORD_ZERO_SHOT_CONDITIONS = [
    ("gold_text", "0shot"),
    ("paddleocr", "0shot"),
    ("easyocr", "0shot"),
    ("tesseract", "0shot"),
]

CORD_FEWSHOT_CONDITIONS = [
    ("gold_text", "1shot"),
    ("gold_text", "2shot"),
    ("gold_text", "3shot"),
    ("paddleocr", "1shot"),
    ("paddleocr", "2shot"),
    ("paddleocr", "3shot"),
]

FEWSHOT_MODEL_NAMES = {m["name"] for m in FEWSHOT_MODELS}


def _validate_model_major_matrix():
    """Guard against accidentally omitting required authoritative conditions."""
    expected_funsd_fewshot = {
        ("gold_text_spatial", "1shot"), ("gold_text_spatial", "2shot"),
        ("gold_text_spatial", "3shot"), ("paddleocr", "1shot"),
        ("paddleocr", "2shot"), ("paddleocr", "3shot"),
    }
    expected_other_fewshot = {
        ("gold_text", "1shot"), ("gold_text", "2shot"), ("gold_text", "3shot"),
        ("paddleocr", "1shot"), ("paddleocr", "2shot"), ("paddleocr", "3shot"),
    }
    assert set(FUNSD_FEWSHOT_CONDITIONS) == expected_funsd_fewshot
    assert set(SROIE_FEWSHOT_CONDITIONS) == expected_other_fewshot
    assert set(CORD_FEWSHOT_CONDITIONS) == expected_other_fewshot
    assert ("gold_text_spatial", "0shot") in FUNSD_ZERO_SHOT_CONDITIONS
    assert len(ALL_MODELS) == 5 and len(FEWSHOT_MODELS) == 2
    print("PASS: authoritative model-major matrix is complete.")
    print("  FUNSD Gold: corrected spatially reconstructed Gold text")
    print("  Quantitative models: 5 (DeepSeek excluded)")
    print("  Few-shot models: Qwen + LLaMA")


_validate_model_major_matrix()


def run_model_major_benchmark(
    include_funsd=True,
    include_sroie=True,
    include_cord=True,
):
    """
    Run/resume the full benchmark one model at a time.

    Authoritative experiment policy:
      - all five valid models: 0-shot on Gold, PaddleOCR, EasyOCR, Tesseract
      - Qwen2.5-7B + LLaMA-3-8B: 1/2/3-shot on Gold + PaddleOCR
      - same policy for FUNSD, SROIE, and CORD

    Cache policy:
      1. verify local-disk headroom before each model;
      2. run every required condition for that model;
      3. delete only that model's local HF cache;
      4. continue to the next model.

    Saved Drive results are never deleted.
    Each runner's RESUME=True logic determines which completed documents
    can be skipped.
    """
    for spec in ALL_MODELS:
        model_name = spec["name"]
        one_model = [spec]

        print("\n" + "#" * 90)
        print(f"MODEL-MAJOR BLOCK: {model_name}")
        print("#" * 90)

        ensure_disk_headroom(
            current_model=model_name,
            min_free_gb=MIN_FREE_DISK_GB,
        )

        try:
            if include_funsd:
                for engine, shot in FUNSD_ZERO_SHOT_CONDITIONS:
                    run_funsd_experiment(engine, shot, one_model)

                if model_name in FEWSHOT_MODEL_NAMES:
                    for engine, shot in FUNSD_FEWSHOT_CONDITIONS:
                        run_funsd_experiment(engine, shot, one_model)

            if include_sroie:
                for engine, shot in SROIE_ZERO_SHOT_CONDITIONS:
                    run_sroie_experiment(engine, shot, one_model)

                if model_name in FEWSHOT_MODEL_NAMES:
                    for engine, shot in SROIE_FEWSHOT_CONDITIONS:
                        run_sroie_experiment(engine, shot, one_model)

            if include_cord:
                for engine, shot in CORD_ZERO_SHOT_CONDITIONS:
                    run_cord_experiment(engine, shot, one_model)

                if model_name in FEWSHOT_MODEL_NAMES:
                    for engine, shot in CORD_FEWSHOT_CONDITIONS:
                        run_cord_experiment(engine, shot, one_model)

        finally:
            # Always release the model's local cache after its block,
            # including if a later condition fails. Saved result files on
            # Drive remain intact and can be resumed on the next run.
            remove_model_cache(model_name)

    print("\nAll requested model-major benchmark blocks completed.")


### Continue/resume the authoritative benchmark

The final execution cell is safe to rerun. Existing completed documents on Google Drive are skipped; incomplete runs continue. FUNSD Gold uses `gold_text_spatial`, the audited corrected training demonstrations are used for FUNSD 1/2/3-shot, and DeepSeek is excluded from the quantitative queue.

After a Colab crash, reconnect Drive, rerun setup/definition cells, then run the final execution cell again.


In [ ]:
# ============================================================
# RUN BENCHMARK — ONLY EXECUTION CELL
# ============================================================
# This is the ONLY cell in the notebook that launches benchmark experiments.
#
# With RESUME=True inside the dataset runners:
#   - completed saved documents are skipped;
#   - incomplete/missing runs continue;
#   - saved Drive results are not deleted.
#
# The model-major scheduler runs one model through all required conditions,
# then removes only that model's local Hugging Face cache before continuing.

run_model_major_benchmark(
    include_funsd=True,
    include_sroie=True,
    include_cord=True,
)


## 10. Notes for reproducibility

- FUNSD Gold uses frozen **spatially reconstructed Gold text** (`funsd_gold_text_spatial`).
- FUNSD few-shot uses the three audited training demonstrations in `funsd_prompt_examples`. The historical `89368010_kvp.json` filename is intentionally mapped to `88547278_88547279.txt`; its 16 KVPs were verified against that annotation.
- DeepSeek is excluded from the final quantitative queue because common-pipeline outputs showed tokenizer artifacts requiring model-specific intervention.
- `RESUME=True` is preserved. Per-document CSV and JSONL writes are flushed and `fsync`'d to reduce loss after Colab interruption.
- Outputs remain under `/content/drive/MyDrive/data/pixels_to_pairs_final_results/`.
- Hugging Face caches are local and can be deleted between model blocks without affecting Drive results.
- Scientific settings remain unchanged: deterministic greedy generation, `MAX_NEW_TOKENS=1024`, existing parser/key normalization/metrics, and no model-specific output repair.
